# Instancia 2 del Poster — Versión Corregida
## Predicción de Aspiraciones Migratorias — Nigeria GHS Wave 5

Constanza Efkhanian, Julián Notario y María Florencia Pascale

**Fuente:** Nigeria General Household Survey, World Bank  
https://microdata.worldbank.org/catalog/6410/study-description  
National Bureau of Statistics (NBS), gobierno federal de Nigeria


### 0 — Bibliotecas y configuración global

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# missingno para heatmap de missings
try:
    import missingno as msno
except ImportError:
    os.system("pip install missingno -q")
    import missingno as msno

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


### Paths

In [2]:
BASE = "/Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/Proyecto_poster/cvs"   
PP   = BASE
PH   = BASE
OUT  = "/Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/Proyecto_poster/diagnostico" 
os.makedirs(OUT, exist_ok=True)


### Estilo global
Principios de Schwabish (2014): reducir clutter, mostrar los datos, integrar texto.

In [3]:
PALETTE   = "#2C3E50"
ACCENT    = "#E74C3C"
SECONDARY = "#3498DB"
GRAY      = "#95A5A6"

sns.set_theme(style="ticks", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor"  : "white",
    "axes.facecolor"    : "white",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.spines.left"  : True,
    "axes.spines.bottom": True,
    "axes.grid"         : True,
    "axes.grid.axis"    : "y",
    "grid.color"        : "#E8E8E8",
    "grid.linewidth"    : 0.6,
    "axes.linewidth"    : 0.8,
    "xtick.major.size"  : 3,
    "ytick.major.size"  : 3,
    "font.family"       : "sans-serif",
    "axes.labelsize"    : 10,
    "axes.titlesize"    : 11,
    "figure.titlesize"  : 13,
})

def save(fig, name):
    path = f"{OUT}/{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  v {name}.png")

def yn_to_bin(s):
    """Convierte '1. YES' / '2. NO' a 1 / 0."""
    return s.map({"1. YES": 1, "2. NO": 0})

def label_bars_h(ax, fmt="{:.1f}%", pad=0.4, fontsize=9):
    """Etiquetas directas en barras horizontales (Schwabish: integrar texto)."""
    for bar in ax.patches:
        w = bar.get_width()
        if abs(w) > 0:
            ax.text(w + pad, bar.get_y() + bar.get_height() / 2,
                    fmt.format(w), va="center", ha="left", fontsize=fontsize)

def label_bars_v(ax, fmt="{:.1f}%", pad=1, fontsize=9):
    """Etiquetas directas en barras verticales."""
    for bar in ax.patches:
        h = bar.get_height()
        if abs(h) > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + pad,
                    fmt.format(h), va="bottom", ha="center",
                    fontsize=fontsize, fontweight="bold")


### SECCIÓN 0 — Cargar archivos con columnas seleccionadas

In [4]:
print("[0] Cargando archivos...")

# -- Demografía individual (sect1) -------------------------
DEMO_COLS = ["hhid", "indiv", "zone",
             "s1q2",   # sexo (MALE/FEMALE)
             "s1q6",   # edad
             "s1q8",   # casado (YES/NO)
             "s1q25",  # actualmente en la escuela
             "s1q28",  # nivel educativo más alto
             "s1q29",  # sector (urbano/rural)
             ]
demo = pd.read_csv(f"{PP}/sect1_plantingw5.csv",
                   usecols=DEMO_COLS, low_memory=False)

# -- Target planting (sect1d) ------------------------------
# CORRECCIÓN: s1dq3b, s1dq5, s1dq7 tienen >29% missing por skip logic;
# se cargan solo para diagnóstico pero no entran como features.
TARGET_COLS_PP = ["hhid", "indiv", "FILTER",
                  "s1dq1",   # desea migrar (variable TARGET principal)
                  "s1dq3a",  # destino dentro del país (diagnóstico)
                  ]
target_pp = pd.read_csv(f"{PP}/sect1d_plantingw5.csv",
                        usecols=TARGET_COLS_PP, low_memory=False)

# -- Remesas (sect1e) --------------------------------------
# CORRECCIÓN: s1eq0a era indicador de proxy-respondent, NO remesas.
# Las variables reales son s1eq1__1 a s1eq1__4.
REMIT_COLS = ["hhid", "indiv",
              "s1eq1__1",  # asistencia monetaria del EXTERIOR
              "s1eq1__2",  # asistencia en especie del EXTERIOR
              "s1eq1__3",  # asistencia monetaria de NIGERIA
              "s1eq1__4",  # asistencia en especie de NIGERIA
              ]
remit = pd.read_csv(f"{PP}/sect1e_plantingw5.csv",
                    usecols=REMIT_COLS, low_memory=False)

# -- Mercado laboral (sect4a) ------------------------------
# s4aq1 varianza cero entre adultos → excluido
# s4aq47_tunit necesario para estandarizar ingresos
LABOR_COLS = ["hhid", "indiv",
              "s4aq4",        # buscó trabajo en los últimos 7 días
              "s4aq46",       # horas por DÍA en el trabajo principal
              "s4aq47",       # ingresos en el trabajo principal
              "s4aq47_tunit", # unidad de tiempo del ingreso
              ]
labor = pd.read_csv(f"{PP}/sect4a_plantingw5.csv",
                    usecols=LABOR_COLS, low_memory=False)

# -- Seguridad alimentaria (sect9, nivel hogar) ------------
FOOD_COLS = ["hhid",
             "s9q1a", "s9q1b", "s9q1c", "s9q1d", "s9q1e",
             "s9q1f", "s9q1g", "s9q1h", "s9q1i", "s9q1j",
             "s9q2a",  # HFIAS score agregado
             "s9q4",   # días sin comer
             ]
food = pd.read_csv(f"{PP}/sect9_plantingw5.csv",
                   usecols=FOOD_COLS, low_memory=False)

# -- Pesos muestrales (secta) ------------------------------
WEIGHTS_COLS = ["hhid", "wt_cross_wave5", "sector"]
weights = pd.read_csv(f"{PP}/secta_plantingw5.csv",
                      usecols=WEIGHTS_COLS, low_memory=False)
weights = weights.rename(columns={"sector": "sector_hh"})

# -- Target harvest (sect3b) -------------------------------
# PPMIGASP_prefilled: aspiración migratoria en harvest (para comparación)
TARGET_COLS_PH = ["hhid", "indiv", "s3bq1", "PPMIGASP_prefilled"]
target_ph = pd.read_csv(f"{PH}/sect3b_harvestw5.csv",
                        usecols=TARGET_COLS_PH, low_memory=False)

# -- Shocks económicos (sect12) ----------------------------
shocks_raw = pd.read_csv(f"{PH}/sect12_harvestw5.csv", low_memory=False)

print("  Archivos cargados correctamente.")
print(f"  demo:       {demo.shape}")
print(f"  target_pp:  {target_pp.shape}")
print(f"  remit:      {remit.shape}")
print(f"  labor:      {labor.shape}")
print(f"  food:       {food.shape}")
print(f"  weights:    {weights.shape}")
print(f"  target_ph:  {target_ph.shape}")
print(f"  shocks_raw: {shocks_raw.shape}")


[0] Cargando archivos...
  Archivos cargados correctamente.
  demo:       (32009, 9)
  target_pp:  (27421, 5)
  remit:      (27421, 6)
  labor:      (27421, 6)
  food:       (4771, 13)
  weights:    (5067, 3)
  target_ph:  (25662, 4)
  shocks_raw: (136735, 20)


### SECCIÓN 1 — Estructura de los archivos

In [5]:
print("[1] Estructura de archivos...")

file_info = {
    "sect1d (target planting)": target_pp,
    "sect1 (demografía)"      : demo,
    "sect1e (remesas)"        : remit,
    "sect4a (mercado laboral)": labor,
    "sect9 (seg. alimentaria)": food,
    "secta (pesos)"           : weights,
    "sect3b (target harvest)" : target_ph,
    "sect12 (shocks)"         : shocks_raw,
}

names = list(file_info.keys())
rows  = [df.shape[0] for df in file_info.values()]
cols_ = [df.shape[1] for df in file_info.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Sección 1 — Estructura de los archivos de entrada", fontweight="bold")

for ax, values, xlabel, title, color in [
    (axes[0], rows,  "Filas",     "Observaciones por archivo",  SECONDARY),
    (axes[1], cols_, "Variables", "Variables seleccionadas",    PALETTE),
]:
    bars = ax.barh(names, values, color=color, alpha=0.85, height=0.55)
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontweight="bold")
    ax.invert_yaxis()
    ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.tick_params(left=False)
    for bar, v in zip(bars, values):
        ax.text(bar.get_width() + max(values) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{v:,}", va="center", fontsize=9)

plt.tight_layout()
save(fig, "01_estructura_archivos")


[1] Estructura de archivos...
  v 01_estructura_archivos.png


### SECCIÓN 2 — Factibilidad del merge

In [6]:
print("[2] Factibilidad del merge...")

indiv_files = {"sect1d": target_pp, "sect1": demo,
               "sect1e": remit, "sect4a": labor, "sect3b": target_ph}
hh_files    = {"sect9": food, "secta": weights}

target_indivs = set(zip(
    target_pp[target_pp["FILTER"] == "1. YES"]["hhid"],
    target_pp[target_pp["FILTER"] == "1. YES"]["indiv"]
))
hh_target = set(target_pp[target_pp["FILTER"] == "1. YES"]["hhid"])

merge_stats = []
for name, df_ in indiv_files.items():
    key     = set(zip(df_["hhid"], df_["indiv"]))
    overlap = len(key & target_indivs)
    merge_stats.append({
        "Archivo"         : name,
        "Nivel"           : "Individuo",
        "Unidades únicas" : len(key),
        "Overlap c/target": overlap,
        "% cobertura"     : round(overlap / len(target_indivs) * 100, 1),
    })
for name, df_ in hh_files.items():
    hh_file  = set(df_["hhid"])
    overlap  = len(hh_file & hh_target)
    merge_stats.append({
        "Archivo"         : name,
        "Nivel"           : "Hogar",
        "Unidades únicas" : len(hh_file),
        "Overlap c/target": overlap,
        "% cobertura"     : round(overlap / len(hh_target) * 100, 1),
    })

merge_df = pd.DataFrame(merge_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 4),
                         gridspec_kw={"width_ratios": [2, 1]})
fig.suptitle(
    "Sección 2 — Factibilidad del merge\n"
    "(universo: individuos con FILTER=YES en sect1d)",
    fontweight="bold")

ax_tbl = axes[0]
ax_tbl.axis("off")
tbl = ax_tbl.table(cellText=merge_df.values, colLabels=merge_df.columns,
                   loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.8)
for j in range(len(merge_df.columns)):
    tbl[0, j].set_facecolor(PALETTE)
    tbl[0, j].set_text_props(color="white", fontweight="bold")
for i in range(1, len(merge_df) + 1):
    pct   = float(merge_df["% cobertura"].iloc[i - 1])
    color = "#27AE60" if pct > 95 else ("#F39C12" if pct > 80 else "#E74C3C")
    tbl[i, 4].set_facecolor(color)
    tbl[i, 4].set_text_props(color="white", fontweight="bold")

ax_bar = axes[1]
colors_cov = ["#27AE60" if p > 95 else ("#F39C12" if p > 80 else "#E74C3C")
              for p in merge_df["% cobertura"]]
bars = ax_bar.barh(merge_df["Archivo"], merge_df["% cobertura"],
                   color=colors_cov, alpha=0.85, height=0.55)
ax_bar.set_xlim(0, 110)
ax_bar.axvline(100, ls="--", color=GRAY, linewidth=0.8)
ax_bar.set_xlabel("% cobertura")
ax_bar.set_title("Cobertura del merge", fontweight="bold")
ax_bar.invert_yaxis()
ax_bar.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax_bar.grid(axis="y", visible=False)
ax_bar.spines["left"].set_visible(False)
ax_bar.tick_params(left=False)
for bar, p in zip(bars, merge_df["% cobertura"]):
    ax_bar.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                f"{p}%", va="center", fontsize=9)

plt.tight_layout()
save(fig, "02_merge_feasibility")


[2] Factibilidad del merge...
  v 02_merge_feasibility.png


### SECCIÓN 3 — Variable target

In [7]:
print("[3] Variable target...")

tgt = target_pp[target_pp["FILTER"] == "1. YES"].copy()
tgt["target"] = yn_to_bin(tgt["s1dq1"])

tgt_demo = tgt.merge(
    demo[["hhid", "indiv", "zone", "s1q2", "s1q6", "s1q28"]],
    on=["hhid", "indiv"], how="left"
)

# Sector urbano/rural desde weights
tgt_demo = tgt_demo.merge(
    weights[["hhid", "sector_hh"]], on="hhid", how="left"
)

tgt_demo["age_group"] = pd.cut(
    tgt_demo["s1q6"],
    bins=[14, 24, 34, 44, 54, 120],
    labels=["15-24", "25-34", "35-44", "45-54", "55+"]
)

global_rate = tgt_demo["target"].mean() * 100

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    f"Sección 3 — Aspiraciones migratorias: "
    f"{global_rate:.1f}% de los adultos 15+ desea migrar",
    fontweight="bold")
axes = axes.flatten()

COLORS_T = {"Desea migrar": SECONDARY, "No desea": ACCENT}

# 3.1 Distribución global
ax = axes[0]
pcts     = [tgt["target"].mean() * 100, (1 - tgt["target"].mean()) * 100]
labels_g = ["Desea migrar", "No desea migrar"]
bars = ax.barh(labels_g, pcts, color=[SECONDARY, ACCENT], alpha=0.88, height=0.45)
ax.set_xlim(0, 120)
ax.set_xlabel("% de adultos 15+")
ax.set_title(f"Distribución global (n={len(tgt):,})", fontweight="bold")
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
for bar, p in zip(bars, pcts):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{p:.1f}%", va="center", fontsize=12, fontweight="bold")

# 3.2 Por sexo
ax = axes[1]
sex_rate = (tgt_demo.groupby("s1q2")["target"]
            .mean().reset_index()
            .rename(columns={"s1q2": "Sexo", "target": "Tasa_YES"}))
bars = ax.bar(sex_rate["Sexo"], sex_rate["Tasa_YES"] * 100,
              color=[SECONDARY, ACCENT], alpha=0.88, width=0.45)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1,
           label=f"Media global ({global_rate:.1f}%)")
ax.set_ylim(0, 100)
ax.set_ylabel("% desea migrar")
ax.set_title("Tasa de aspiración por sexo", fontweight="bold")
ax.legend(fontsize=8, frameon=False)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
label_bars_v(ax)

# 3.3 Por zona geográfica
ax = axes[2]
zone_rate   = tgt_demo.groupby("zone")["target"].mean().sort_values()
short_zones = [str(z).split(". ")[-1] for z in zone_rate.index]
bars = ax.barh(short_zones, zone_rate.values * 100,
               color=PALETTE, alpha=0.82, height=0.55)
ax.axvline(global_rate, ls="--", color=ACCENT, linewidth=1, alpha=0.7)
ax.set_xlabel("% desea migrar")
ax.set_title("Tasa por zona geográfica", fontweight="bold")
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
for bar, v in zip(bars, zone_rate.values * 100):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{v:.1f}%", va="center", fontsize=9)

# 3.4 Por grupo de edad
ax = axes[3]
age_rate = tgt_demo.groupby("age_group", observed=True)["target"].agg(["mean","count"])
bars = ax.bar(age_rate.index.astype(str), age_rate["mean"] * 100,
              color=SECONDARY, alpha=0.82)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1, alpha=0.7)
ax.set_ylim(0, 100)
ax.set_ylabel("% desea migrar")
ax.set_title("Tasa de aspiración por grupo de edad", fontweight="bold")
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
for bar, (_, row) in zip(bars, age_rate.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{row['mean']*100:.1f}%", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
    ax.text(bar.get_x() + bar.get_width()/2, 2,
            f"n={int(row['count'])}", ha="center", va="bottom",
            fontsize=7, color="white", fontweight="bold")

# 3.5 Por sector (urbano/rural)
ax = axes[4]
if "sector_hh" in tgt_demo.columns:
    sec_rate = (tgt_demo.groupby("sector_hh")["target"]
                .agg(["mean","count"]).dropna())
    short_sec = [str(s).split(". ")[-1] for s in sec_rate.index]
    bars = ax.bar(short_sec, sec_rate["mean"] * 100,
                  color=PALETTE, alpha=0.82, width=0.45)
    ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1)
    ax.set_ylim(0, 100)
    ax.set_ylabel("% desea migrar")
    ax.set_title("Tasa por sector (urbano/rural)", fontweight="bold")
    ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
    ax.grid(axis="x", visible=False)
    label_bars_v(ax)
else:
    ax.set_visible(False)

# 3.6 Transición planting → harvest
ax = axes[5]
both = tgt_demo.merge(
    target_ph[["hhid", "indiv", "s3bq1"]], on=["hhid", "indiv"], how="inner"
)
both["pp"] = yn_to_bin(both["s1dq1"])
both["ph"] = yn_to_bin(both["s3bq1"])
trans = pd.crosstab(
    both["pp"].map({1: "Sí (planting)", 0: "No (planting)"}),
    both["ph"].map({1: "Sí (harvest)",  0: "No (harvest)"}),
    normalize="index"
) * 100
sns.heatmap(trans, annot=True, fmt=".1f", cmap="Blues",
            ax=ax, cbar=False, linewidths=0.4, linecolor="white",
            annot_kws={"size": 11, "weight": "bold"})
ax.set_title("Transición aspiraciones\nplanting → harvest (%)", fontweight="bold")
ax.set_ylabel("Planting")
ax.set_xlabel("Harvest")
ax.text(0, -0.18, "Valores: % de fila",
        transform=ax.transAxes, fontsize=8, color=GRAY)

plt.tight_layout()
save(fig, "03_target_variable")


[3] Variable target...
  v 03_target_variable.png


### SECCIÓN 4 — Dataset fusionado: missings y correlaciones

In [8]:
print("[4] Dataset fusionado y missings...")

base = tgt[["hhid", "indiv", "target"]].copy()
base = base.merge(
    demo[["hhid", "indiv", "zone", "s1q2", "s1q6", "s1q8", "s1q25", "s1q28"]],
    on=["hhid", "indiv"], how="left")
base = base.merge(
    remit[["hhid", "indiv", "s1eq1__1", "s1eq1__2", "s1eq1__3", "s1eq1__4"]],
    on=["hhid", "indiv"], how="left")
base = base.merge(
    labor[["hhid", "indiv", "s4aq4", "s4aq46", "s4aq47", "s4aq47_tunit"]],
    on=["hhid", "indiv"], how="left")
base = base.merge(food, on="hhid", how="left")
base = base.merge(weights[["hhid", "wt_cross_wave5"]], on="hhid", how="left")

# target_harvest se conserva aparte (no es feature, es segunda medición del target)
target_harvest_df = target_ph[["hhid","indiv","s3bq1"]].rename(
    columns={"s3bq1":"target_harvest"})

print(f"  Dataset fusionado: {base.shape[0]:,} filas x {base.shape[1]} columnas")

def tag(col):
    if col in ["hhid", "indiv", "target", "target_harvest"]: return "IDs / Target"
    if col.startswith("s1dq"):  return "Aspiración (sect1d)"
    if col in ["s1q2","s1q6","s1q8","s1q25","s1q28","zone"]: return "Demografía (sect1)"
    if col.startswith("s1eq"): return "Remesas (sect1e)"
    if col.startswith("s4aq"): return "Laboral (sect4a)"
    if col.startswith("s9q"):  return "Seg. alim. (sect9)"
    if col.startswith("wt"):   return "Pesos (secta)"
    return "Otro"

miss_df = (base.isnull().mean() * 100).reset_index()
miss_df.columns = ["variable", "pct_missing"]
miss_df["grupo"] = miss_df["variable"].apply(tag)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Sección 4a — Valores faltantes en el dataset fusionado", fontweight="bold")

ax = axes[0]
miss_mat   = base.isnull().astype(int)
sample_idx = np.random.choice(miss_mat.index, size=min(1000, len(miss_mat)), replace=False)
sns.heatmap(miss_mat.loc[sample_idx].T,
            cmap=["#27AE60", "#E74C3C"], cbar=False,
            ax=ax, yticklabels=True, xticklabels=False, linewidths=0)
ax.set_title("Patrón de missings (muestra 1,000 obs.)\n"
             "Verde = presente  |  Rojo = faltante", fontweight="bold")
ax.set_xlabel("Observaciones")
ax.tick_params(axis="y", labelsize=7)

ax2 = axes[1]
miss_with = miss_df[miss_df["pct_missing"] > 0].sort_values("pct_missing")
colors_miss = miss_with["pct_missing"].apply(
    lambda x: "#27AE60" if x < 10 else ("#F39C12" if x < 40 else "#E74C3C"))
ax2.barh(miss_with["variable"], miss_with["pct_missing"],
         color=colors_miss, alpha=0.85, height=0.65)
ax2.axvline(10, ls="--", color="#F39C12", linewidth=0.9, label="10%")
ax2.axvline(40, ls="--", color="#E74C3C", linewidth=0.9, label="40%")
ax2.set_xlabel("% valores faltantes")
ax2.set_title("% missing por variable\n"
              "Verde <10%  |  Naranja 10-40%  |  Rojo >40%", fontweight="bold")
ax2.legend(fontsize=9, frameon=False)
ax2.tick_params(axis="y", labelsize=8)
ax2.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax2.grid(axis="y", visible=False)
ax2.spines["left"].set_visible(False)
ax2.tick_params(left=False)

plt.tight_layout()
save(fig, "04a_missing_values")


[4] Dataset fusionado y missings...
  Dataset fusionado: 16,590 filas x 30 columnas
  v 04a_missing_values.png


In [9]:
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Sección 4b — % missing promedio por grupo de variables", fontweight="bold")
group_miss = miss_df.groupby("grupo")["pct_missing"].mean().sort_values()
colors_g   = group_miss.apply(
    lambda x: "#27AE60" if x < 10 else ("#F39C12" if x < 40 else "#E74C3C"))
bars = ax.barh(group_miss.index, group_miss.values,
               color=colors_g, alpha=0.85, height=0.55)
ax.axvline(10, ls="--", color="#F39C12", linewidth=0.9)
ax.axvline(40, ls="--", color="#E74C3C", linewidth=0.9)
ax.set_xlabel("% missing promedio")
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.1f}%", va="center", fontsize=10)
plt.tight_layout()
save(fig, "04b_missing_por_grupo")


  v 04b_missing_por_grupo.png


### SECCIÓN 4c — Heatmap de missing values (missingno)

In [10]:
print("[4c] Heatmap de missing values (con etiquetas legibles)...")

# ── MAPA DE ETIQUETAS LEGIBLES ────────────────────────────────────────
var_labels = {
    "s4aq47_tunit" : "Unidad tiempo del ingreso",
    "s4aq47"       : "Ingresos (trabajo ppal.)",
    "s4aq46"       : "Horas por día (trabajo)",
    "s1q28"        : "Nivel educativo",
    "wt_cross_wave5": "Peso muestral",
    "s9q2a"        : "HFIAS score",
    "s4aq4"        : "Buscó trabajo (7 días)",
}

# Variables con missings > 0 (excluyendo IDs)
cols_con_miss = [c for c in base.columns
                 if base[c].isnull().mean() > 0 and c not in ["hhid","indiv"]]

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(cols_con_miss) * 0.5 + 2)))
fig.suptitle("Sección 4c — Heatmap de Missing Values", fontweight="bold")

# Panel izq: % missing por variable (etiquetas legibles)
ax_left = axes[0]
miss_pct = base[cols_con_miss].isnull().mean() * 100
miss_pct = miss_pct.sort_values(ascending=True)

# Reemplazar nombres técnicos por etiquetas legibles
readable_idx = [var_labels.get(c, c) for c in miss_pct.index]

colors_hm = ["#E74C3C" if v > 40 else "#F39C12" if v > 10 else "#27AE60"
             for v in miss_pct.values]
bars = ax_left.barh(readable_idx, miss_pct.values,
                    color=colors_hm, alpha=0.85, height=0.6)
ax_left.axvline(10, ls="--", color="#F39C12", linewidth=1, label="10%")
ax_left.axvline(40, ls="--", color="#E74C3C", linewidth=1, label="40%")
ax_left.set_xlabel("% valores faltantes")
ax_left.set_title("Porcentaje missing por variable\n"
                  "Verde <10% | Naranja 10-40% | Rojo >40%", fontweight="bold")
ax_left.legend(fontsize=8, frameon=False)
ax_left.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax_left.grid(axis="y", visible=False)
ax_left.spines["left"].set_visible(False)
ax_left.tick_params(left=False, labelsize=9)
for bar in bars:
    w = bar.get_width()
    ax_left.text(w + 0.5, bar.get_y() + bar.get_height() / 2,
                 f"{w:.1f}%", va="center", ha="left", fontsize=8)

# Panel der: heatmap de correlación de missings (con etiquetas legibles)
ax_right = axes[1]
miss_matrix = base[cols_con_miss].isnull().astype(float)
corr_miss   = miss_matrix.corr()

# Renombrar filas y columnas con etiquetas legibles
corr_miss.index   = [var_labels.get(c, c) for c in corr_miss.index]
corr_miss.columns = [var_labels.get(c, c) for c in corr_miss.columns]

mask_diag = np.eye(len(corr_miss), dtype=bool)
sns.heatmap(
    corr_miss, ax=ax_right,
    cmap="RdYlGn_r", center=0, vmin=-1, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    mask=mask_diag,
    linewidths=0.3, linecolor="white",
    cbar_kws={"shrink": 0.7, "label": "Corr. de missings"}
)
ax_right.set_title("Correlación entre patrones de missing\n"
                   "(valores altos = faltan juntos)", fontweight="bold")
ax_right.tick_params(axis="x", rotation=40, labelsize=8)
ax_right.tick_params(axis="y", rotation=0,  labelsize=8)

plt.tight_layout()
save(fig, "04c_heatmap_missings")
print("  v 04c_heatmap_missings.png")


[4c] Heatmap de missing values (con etiquetas legibles)...
  v 04c_heatmap_missings.png
  v 04c_heatmap_missings.png


### SECCIÓN 4d — Correlaciones entre features

In [11]:
print("[4d] Correlaciones con target...")

rename_map = {
    "target"    : "TARGET",
    "s1eq1__1"  : "rem_ext_mon",   "s1eq1__2": "rem_ext_ink",
    "s1eq1__3"  : "rem_int_mon",   "s1eq1__4": "rem_int_ink",
    "s1q6"      : "edad",          "s1q8"   : "casado",
    "s1q25"     : "en_escuela",
    "s4aq4"     : "busca_empleo",  "s4aq46" : "horas_x_dia",
    "s4aq47"    : "ingresos",
    "s9q1a"     : "HFIAS_1",       "s9q1b"  : "HFIAS_2",
    "s9q1c"     : "HFIAS_3",       "s9q1d"  : "HFIAS_4",
    "s9q1e"     : "HFIAS_5",       "s9q1f"  : "HFIAS_6",
    "s9q1g"     : "HFIAS_7",       "s9q1h"  : "HFIAS_8",
    "s9q1i"     : "HFIAS_9",       "s9q1j"  : "HFIAS_10",
    "s9q2a"     : "HFIAS_score",   "s9q4"   : "sin_comer",
}

corr_df = base.copy()
yn_cols_corr = ["s4aq4",
                "s9q1a","s9q1b","s9q1c","s9q1d","s9q1e",
                "s9q1f","s9q1g","s9q1h","s9q1i","s9q1j","s9q4",
                "s1eq1__1","s1eq1__2","s1eq1__3","s1eq1__4"]
for c in yn_cols_corr:
    if c in corr_df.columns:
        corr_df[c] = yn_to_bin(corr_df[c])

num_cols = corr_df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ["hhid","indiv","wt_cross_wave5"]]
corr_mat = (corr_df[num_cols].corr()
            .rename(index=rename_map, columns=rename_map)
            .dropna(axis=0, how="all").dropna(axis=1, how="all"))

fig, ax = plt.subplots(figsize=(16, 13))
fig.suptitle("Sección 4d — Matriz de correlación (Pearson)", fontweight="bold")
mask      = np.triu(np.ones_like(corr_mat, dtype=bool))
annot_arr = corr_mat.map(lambda v: f"{v:.2f}" if abs(v) >= 0.10 else "")
sns.heatmap(corr_mat, mask=mask, annot=annot_arr, fmt="",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.2, annot_kws={"size": 8},
            cbar_kws={"shrink": 0.7, "label": "Correlación de Pearson"})
ax.set_title("Triángulo inferior  |  se muestran |r| >= 0.10",
             fontsize=10, pad=10)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
save(fig, "04d_correlacion")


[4d] Correlaciones con target...
  v 04d_correlacion.png


In [12]:
target_corr = corr_mat["TARGET"].drop("TARGET").sort_values().dropna()

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle("Sección 4e — Correlación de cada variable con la aspiración migratoria",
             fontweight="bold")
colors_bar = [ACCENT if v < 0 else SECONDARY for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors_bar, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlación de Pearson con TARGET")
ax.set_title("Azul = correlación positiva  |  Rojo = negativa", fontsize=9)
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
for v, y in zip(target_corr.values, range(len(target_corr))):
    ha  = "left"  if v >= 0 else "right"
    pad = 0.003   if v >= 0 else -0.003
    ax.text(v + pad, y, f"{v:.2f}", va="center", ha=ha, fontsize=8)
plt.tight_layout()
save(fig, "04e_corr_con_target")


  v 04e_corr_con_target.png


### SECCIÓN 5 — Distribuciones de features clave vs. target

In [13]:
print("[5] Distribuciones por target...")

plot_df = corr_df.copy()
plot_df["target_label"] = plot_df["target"].map({1: "Desea migrar", 0: "No desea"})
COLORS_T = {"Desea migrar": SECONDARY, "No desea": ACCENT}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    "Sección 5 — Distribuciones de variables clave por aspiración migratoria",
    fontweight="bold", y=1.02)
axes = axes.flatten()

# 5.1 Edad
ax = axes[0]
for lbl, grp in plot_df.groupby("target_label"):
    ages = grp["s1q6"].dropna()
    ages = ages[(ages >= 15) & (ages <= 80)]
    ax.hist(ages, bins=20, alpha=0.55, label=lbl, density=True,
            color=COLORS_T[lbl], edgecolor="white", linewidth=0.4)
    ax.axvline(ages.mean(), color=COLORS_T[lbl], linestyle="--",
               linewidth=1.2, alpha=0.85)
g1 = plot_df[plot_df["target"] == 1]["s1q6"].dropna()
g0 = plot_df[plot_df["target"] == 0]["s1q6"].dropna()
_, p = stats.ttest_ind(g1, g0)
pval = "<0.001" if p < 0.001 else f"{p:.3f}"
ax.set_title(f"Distribución de edad  (t-test p={pval})", fontweight="bold")
ax.set_xlabel("Años")
ax.set_ylabel("Densidad")
ax.legend(fontsize=8, frameon=False)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)

# 5.2 Score HFIAS
ax = axes[1]
hfias_rate = plot_df.groupby("s9q2a")["target"].mean() * 100
ax.bar(hfias_rate.index, hfias_rate.values, color=PALETTE, alpha=0.85, width=0.8)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1,
           label=f"Tasa global ({global_rate:.1f}%)")
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(["Seguro", "Leve", "Moderado", "Severo"], fontsize=9)
ax.set_ylabel("% desea migrar")
ax.set_xlabel("Nivel de inseguridad alimentaria (HFIAS)")
ax.set_title("Aspiración por inseguridad alimentaria", fontweight="bold")
ax.set_ylim(0, 100)
ax.legend(fontsize=8, frameon=False)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
label_bars_v(ax)

# 5.3 Ingresos laborales (log)
ax = axes[2]
for lbl, grp in plot_df.groupby("target_label"):
    inc = grp["s4aq47"].dropna()
    inc = inc[(inc > 0) & (inc < inc.quantile(0.99))]
    if len(inc) > 10:
        log_inc = np.log1p(inc)
        ax.hist(log_inc, bins=25, alpha=0.55, label=lbl,
                density=True, color=COLORS_T[lbl], edgecolor="white", linewidth=0.4)
        ax.axvline(log_inc.mean(), color=COLORS_T[lbl],
                   linestyle="--", linewidth=1.2, alpha=0.85)
ax.set_title("Log(Ingresos laborales)", fontweight="bold")
ax.set_xlabel("Log(1 + ingresos, NGN)  [p99 truncado]")
ax.set_ylabel("Densidad")
ax.legend(fontsize=8, frameon=False)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)

# 5.4 Nivel educativo (top/bottom 15 categorías para legibilidad)
ax = axes[3]
edu_rate = (plot_df.groupby("s1q28")["target"]
            .mean().reset_index()
            .rename(columns={"target": "rate", "s1q28": "edu"}))
edu_rate["edu_short"] = edu_rate["edu"].astype(str).str.slice(0, 22)
edu_rate = edu_rate.sort_values("rate")
# CORRECCIÓN: limitar a 20 categorías para evitar gráfico ilegible
edu_top = pd.concat([edu_rate.head(10), edu_rate.tail(10)])
ax.barh(edu_top["edu_short"], edu_top["rate"] * 100,
        color=PALETTE, alpha=0.82, height=0.65)
ax.axvline(global_rate, ls="--", color=ACCENT, linewidth=1, alpha=0.7)
ax.set_xlabel("% desea migrar")
ax.set_title("Tasa de aspiración por\nnivel educativo (top/bottom 10)", fontweight="bold")
ax.tick_params(axis="y", labelsize=7.5)
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)

# 5.5 Remesas externas
ax = axes[4]
# CORRECCIÓN: usar remesas_exterior calculada desde s1eq1__1 y s1eq1__2
ext_tmp = yn_to_bin(plot_df["s1eq1__1"].astype(str)).combine(
    yn_to_bin(plot_df["s1eq1__2"].astype(str)), max)
plot_df["_rem_ext"] = ext_tmp
remit_rate = plot_df.groupby("_rem_ext")["target"].mean()
labels_r = ["No recibe\nremesas ext.", "Recibe\nremesas ext."]
vals_r   = [remit_rate.get(0, np.nan) * 100, remit_rate.get(1, np.nan) * 100]
vals_r   = [v for v in vals_r if not np.isnan(v)]
lbl_r    = [labels_r[i] for i, v in enumerate(
             [remit_rate.get(0, np.nan), remit_rate.get(1, np.nan)])
            if not np.isnan(v)]
bars = ax.bar(lbl_r, vals_r, color=[ACCENT, SECONDARY][:len(lbl_r)],
              alpha=0.88, width=0.4)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1)
ax.set_ylim(0, 100)
ax.set_ylabel("% desea migrar")
ax.set_title("Tasa por recepción de remesas externas", fontweight="bold")
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
label_bars_v(ax)

# 5.6 Situación laboral
ax = axes[5]
emp_tmp = yn_to_bin(plot_df["s4aq4"].astype(str))
plot_df["_busca"] = emp_tmp
emp_rate = plot_df.groupby("_busca")["target"].mean()
labels_e = ["No busca empleo", "Busca empleo"]
vals_e   = [emp_rate.get(0, np.nan) * 100, emp_rate.get(1, np.nan) * 100]
vals_ok  = [(l, v) for l, v in zip(labels_e, vals_e) if not np.isnan(v)]
if vals_ok:
    lbls, vals = zip(*vals_ok)
    bars = ax.bar(lbls, vals, color=[PALETTE, SECONDARY], alpha=0.88, width=0.4)
    ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1)
    ax.set_ylim(0, 100)
    ax.set_ylabel("% desea migrar")
    ax.set_title("Tasa por búsqueda de empleo\n(últimos 7 días)", fontweight="bold")
    ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
    ax.grid(axis="x", visible=False)
    label_bars_v(ax)

plt.tight_layout()
save(fig, "05_distribuciones_features")


[5] Distribuciones por target...
  v 05_distribuciones_features.png


### SECCIÓN 6 — Shocks económicos (sect12)

In [14]:
print("[6] Shocks económicos...")

shocks_raw["affected"] = (shocks_raw["s12q1"] == "1. YES").astype(int)
shock_wide = (
    shocks_raw.pivot_table(
        index="hhid", columns="shock_cd",
        values="affected", aggfunc="max", fill_value=0
    ).reset_index()
)
shock_wide.columns.name = None
shock_wide.columns = (
    ["hhid"] + [
        c.split(". ", 1)[-1][:30] if isinstance(c, str) and ". " in c else str(c)
        for c in shock_wide.columns[1:]
    ]
)

shock_prev   = shock_wide.drop(columns="hhid").mean().sort_values(ascending=True)
shock_target = base[["hhid", "target"]].merge(shock_wide, on="hhid", how="left")
shock_target["target"] = pd.to_numeric(shock_target["target"], errors="coerce")
shock_corr = (
    shock_target.drop(columns=["hhid"]).corr()["target"].drop("target").sort_values()
)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle(
    "Sección 6a — Shocks económicos: prevalencia y correlación con aspiraciones",
    fontweight="bold")

ax = axes[0]
colors_shock = [ACCENT if v > 0.15 else SECONDARY for v in shock_prev.values]
bars = ax.barh(shock_prev.index, shock_prev.values * 100,
               color=colors_shock, alpha=0.85, height=0.65)
ax.set_xlabel("% hogares afectados")
ax.set_title("Prevalencia de shocks\n(rojo = afecta a >15% de hogares)",
             fontweight="bold")
ax.tick_params(axis="y", labelsize=8)
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
for bar in bars:
    if bar.get_width() > 1:
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f"{bar.get_width():.1f}%", va="center", fontsize=8)

ax = axes[1]
colors_c = [SECONDARY if v >= 0 else ACCENT for v in shock_corr.values]
ax.barh(shock_corr.index, shock_corr.values, color=colors_c, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlación con aspiración migratoria")
ax.set_title("Correlación de cada shock con el TARGET\n"
             "Azul = positiva  |  Rojo = negativa", fontweight="bold")
ax.tick_params(axis="y", labelsize=8)
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)

plt.tight_layout()
save(fig, "06a_shocks_economicos")


[6] Shocks económicos...
  v 06a_shocks_economicos.png


In [15]:
shock_wide_sub             = shock_wide.copy()
shock_wide_sub["n_shocks"] = shock_wide_sub.drop(columns="hhid").sum(axis=1)
shock_index = (
    base[["hhid", "target"]]
    .merge(shock_wide_sub[["hhid", "n_shocks"]], on="hhid", how="left")
)
shock_agg = (shock_index.groupby("n_shocks")["target"]
             .agg(["mean", "count"]).reset_index())

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Sección 6b — A más shocks acumulados, ¿mayor aspiración migratoria?",
             fontweight="bold")
bars = ax.bar(shock_agg["n_shocks"], shock_agg["mean"] * 100,
              color=PALETTE, alpha=0.82, width=0.7)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1,
           label=f"Media global ({global_rate:.1f}%)")
ax.set_xlabel("Número de shocks distintos que afectaron al hogar")
ax.set_ylabel("% desea migrar")
ax.set_ylim(0, 100)
ax.legend(fontsize=9, frameon=False)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
for bar, (_, row) in zip(bars, shock_agg.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{row['mean']*100:.1f}%", ha="center", va="bottom",
            fontsize=8.5, fontweight="bold")
    if row["count"] > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, 2,
                f"n={int(row['count'])}", ha="center", va="bottom",
                fontsize=7, color="white", fontweight="bold")
plt.tight_layout()
save(fig, "06b_shock_index")


  v 06b_shock_index.png


### SECCIÓN 7 — Tabla resumen de variables para data cleaning

In [16]:
print("[7] Tabla resumen de decisiones...")

# Agregar target_harvest al base temporal para el resumen completo
base_full = base.merge(target_harvest_df, on=["hhid","indiv"], how="left")

summary_rows = []
for col in base_full.columns:
    if col in ["hhid", "indiv"]:
        continue
    miss  = base_full[col].isnull().mean() * 100
    dtype = str(base_full[col].dtype)
    nuniq = base_full[col].nunique(dropna=True)
    if miss > 40:
        rec = "REVISAR - >40% missing"
    elif miss > 10:
        rec = "Imputar (mediana/moda)"
    elif nuniq <= 1:
        rec = "DROP - varianza cero"
    elif col.endswith("_os"):
        rec = "DROP - campo open-ended"
    else:
        rec = "Usar"
    summary_rows.append({
        "Variable"    : col,
        "Interpretado": rename_map.get(col, col),
        "Grupo"       : tag(col),
        "% Missing"   : round(miss, 1),
        "Dtype"       : dtype,
        "N únicos"    : nuniq,
        "Rec."        : rec,
    })

summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv(f"{OUT}/07_summary_tabla.csv", index=False)
print("  OK 07_summary_tabla.csv")

n_per_page = 30
pages = [summary_table.iloc[i:i + n_per_page]
         for i in range(0, len(summary_table), n_per_page)]

REC_COLORS = {
    "Usar"                  : "#D5F5E3",
    "DROP - varianza cero"  : "#FADBD8",
    "DROP - campo open-ended": "#FADBD8",
    "REVISAR - >40% missing": "#FDEBD0",
    "Imputar (mediana/moda)": "#D6EAF8",
}

for pg_idx, page in enumerate(pages):
    fig, ax = plt.subplots(figsize=(20, len(page) * 0.45 + 2))
    fig.suptitle(
        f"Sección 7 — Resumen de variables para data cleaning "
        f"(pág. {pg_idx+1}/{len(pages)})",
        fontsize=13, fontweight="bold")
    ax.axis("off")
    tbl = ax.table(cellText=page.values, colLabels=page.columns,
                   loc="center", cellLoc="left")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.scale(1, 1.5)
    for j in range(len(page.columns)):
        tbl[0, j].set_facecolor(PALETTE)
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    rec_idx = list(page.columns).index("Rec.")
    for i in range(1, len(page) + 1):
        val   = page["Rec."].iloc[i - 1]
        color = REC_COLORS.get(val, "white")
        tbl[i, rec_idx].set_facecolor(color)
    plt.tight_layout()
    save(fig, f"07_summary_tabla_p{pg_idx+1}")


[7] Tabla resumen de decisiones...
  OK 07_summary_tabla.csv
  v 07_summary_tabla_p1.png


### SECCIÓN 7b — Resumen final del diagnóstico

In [17]:
print("\n" + "="*60)
print("RESUMEN DEL DIAGNÓSTICO")
print("="*60)
n_total    = base.shape[0]
n_complete = base.dropna().shape[0]
print(f"  Dataset fusionado : {n_total:,} filas x {base.shape[1]} columnas")
print(f"  Target YES        : {base['target'].mean()*100:.1f}%")
print(f"  Sin ningún missing: {n_complete:,} obs ({n_complete/n_total*100:.1f}%)")
print(f"  Vars sin missings : {(base.isnull().sum()==0).sum()}")
print(f"  Vars >40% missing : {(base.isnull().mean()>0.4).sum()}")
miss_mid = ((base.isnull().mean()>0.1)&(base.isnull().mean()<=0.4)).sum()
print(f"  Vars 10-40% miss. : {miss_mid}")
print(f"  Vars <10% miss.   : {(base.isnull().mean()<0.1).sum()}")
print(f"\nArchivos generados en: {OUT}/")
print("="*60)



RESUMEN DEL DIAGNÓSTICO
  Dataset fusionado : 16,590 filas x 30 columnas
  Target YES        : 71.0%
  Sin ningún missing: 6,462 obs (39.0%)
  Vars sin missings : 23
  Vars >40% missing : 3
  Vars 10-40% miss. : 1
  Vars <10% miss.   : 26

Archivos generados en: /Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/Proyecto_poster/diagnostico/


---
## SECCIÓN 8 — Data Cleaning: Dataset Definitivo para ML

**Decisiones de limpieza basadas en el diagnóstico:**

| Paso | Acción | Justificación |
|------|--------|---------------|
| Drop proxy indicator | `s1eq0a` | Era indicador de respondente, no remesas |
| Drop varianza cero | `s4aq1` | Filtro de edad, constante entre adultos |
| Fix remesas | `s1eq1__1` a `s1eq1__4` | Variables reales de recepción de remesas |
| Estandarizar ingresos | `s4aq47` → `ingreso_mensual` | Unidades heterogéneas (hora/día/mes/año) |
| Imputar educación | `s1q28` → mediana ordinal | 30% missing, distribución ordinal |
| Imputar remesas | `remesas_*` → 0 | Missing implica no recibe |
| Imputar ingresos | `ingreso_mensual` → 0 | Missing implica sin empleo formal |
| Encoding YES/NO | YES=1, NO=0 | Requerimiento de sklearn |
| Encoding zona | zona → dummies (drop_first) | Variable categórica nominal |
| Encoding edu | `s1q28` → ordinal numérico | Variable categórica ordinal |
| Agregar shocks | Pivot `sect12` a wide + índice | 29 tipos de shock por hogar |
| Agregar target_harvest | `s3bq1` → `target_harvest` | Segunda medición del target |


#### 8.1 — Base limpia: partir de tgt + merges corregidos

In [18]:
print("\n[8] DATA CLEANING — construyendo dataset definitivo...")
print("  [8.1] Construyendo base limpia...")

df = tgt[["hhid", "indiv", "target"]].copy()

# Demografía
df = df.merge(
    demo[["hhid","indiv","zone","s1q2","s1q6","s1q8","s1q25","s1q28"]],
    on=["hhid","indiv"], how="left")

# Remesas reales (s1eq1__1 a __4)
df = df.merge(
    remit[["hhid","indiv","s1eq1__1","s1eq1__2","s1eq1__3","s1eq1__4"]],
    on=["hhid","indiv"], how="left")

# Mercado laboral (sin s4aq1, con tunit)
df = df.merge(
    labor[["hhid","indiv","s4aq4","s4aq46","s4aq47","s4aq47_tunit"]],
    on=["hhid","indiv"], how="left")

# Seguridad alimentaria (nivel hogar)
df = df.merge(food, on="hhid", how="left")

# Pesos muestrales
df = df.merge(weights[["hhid","wt_cross_wave5"]], on="hhid", how="left")

# Target harvest (segunda medición - NO es feature)
df = df.merge(
    target_ph[["hhid","indiv","s3bq1"]].rename(columns={"s3bq1":"target_harvest"}),
    on=["hhid","indiv"], how="left")

print(f"    Base inicial: {df.shape[0]:,} filas × {df.shape[1]} columnas")



[8] DATA CLEANING — construyendo dataset definitivo...
  [8.1] Construyendo base limpia...
    Base inicial: 16,590 filas × 31 columnas


#### 8.2 — Convertir YES/NO a binario (1/0)

In [19]:
print("  [8.2] Convirtiendo categóricas YES/NO a binario...")

yn_features = [
    "s1q8", "s1q25",
    "s4aq4",
    "s1eq1__1","s1eq1__2","s1eq1__3","s1eq1__4",
    "s9q1a","s9q1b","s9q1c","s9q1d","s9q1e",
    "s9q1f","s9q1g","s9q1h","s9q1i","s9q1j","s9q4",
    "target_harvest",
]
for c in yn_features:
    if c in df.columns:
        df[c] = yn_to_bin(df[c])

# Sexo: MALE=1, FEMALE=0
df["s1q2"] = df["s1q2"].map({"1. MALE": 1, "2. FEMALE": 0})
print(f"    {len(yn_features) + 1} variables convertidas")


  [8.2] Convirtiendo categóricas YES/NO a binario...
    20 variables convertidas


#### 8.3 — Features de remesas (consolidar en 2 binarias)

In [20]:
print("  [8.3] Construyendo features de remesas...")

# Remesas del exterior: 1 si recibe monetaria O en especie desde fuera
ext = df[["s1eq1__1","s1eq1__2"]]
df["remesas_exterior"] = ext.max(axis=1)
df.loc[ext.isnull().all(axis=1), "remesas_exterior"] = np.nan

# Remesas internas: 1 si recibe monetaria O en especie desde Nigeria
intr = df[["s1eq1__3","s1eq1__4"]]
df["remesas_interior"] = intr.max(axis=1)
df.loc[intr.isnull().all(axis=1), "remesas_interior"] = np.nan

df = df.drop(columns=["s1eq1__1","s1eq1__2","s1eq1__3","s1eq1__4"])

print(f"    remesas_exterior: {df['remesas_exterior'].mean()*100:.1f}% reciben")
print(f"    remesas_interior: {df['remesas_interior'].mean()*100:.1f}% reciben")


  [8.3] Construyendo features de remesas...
    remesas_exterior: 2.5% reciben
    remesas_interior: 19.5% reciben


#### 8.4 — Estandarizar ingresos a base mensual

In [21]:
print("  [8.4] Estandarizando ingresos a base mensual...")

# s4aq47_tunit: 1=hora, 2=día, 3=semana, 4=quincena,
#               5=mes, 6=trimestre, 7=semestre, 8=año
tunit_a_mensual = {1: 176, 2: 22, 3: 4.33, 4: 2.17,
                    5: 1,   6: 1/3, 7: 1/6,  8: 1/12}

df["ingreso_mensual"] = (
    df["s4aq47"]
    * df["s4aq47_tunit"].map(tunit_a_mensual)
)

# Indicador binario: ¿tiene empleo con ingreso registrado?
df["tiene_empleo"] = df["ingreso_mensual"].notna().astype(int)

# Winsorizar outliers extremos (p99)
p99    = df["ingreso_mensual"].quantile(0.99)
n_wins = (df["ingreso_mensual"] > p99).sum()
df["ingreso_mensual"] = df["ingreso_mensual"].clip(upper=p99)

# Drop columnas originales de mercado laboral no estandarizadas
df = df.drop(columns=["s4aq46", "s4aq47", "s4aq47_tunit"])

print(f"    {df['tiene_empleo'].sum():,} individuos con empleo formal ({df['tiene_empleo'].mean()*100:.1f}%)")
print(f"    Ingreso mensual mediano (trabajadores): "
      f"NGN {df.loc[df['tiene_empleo']==1,'ingreso_mensual'].median():,.0f}")
print(f"    Winsorizados en p99 = NGN {p99:,.0f}: {n_wins} obs")


  [8.4] Estandarizando ingresos a base mensual...
    0 individuos con empleo formal (0.0%)
    Ingreso mensual mediano (trabajadores): NGN nan
    Winsorizados en p99 = NGN nan: 0 obs


#### 8.5 — Agregar shocks económicos (sect12)

In [22]:
print("  [8.5] Integrando shocks económicos (sect12)...")

all_shock_cols = [c for c in shock_wide.columns if c != "hhid"]

# Índice de shocks (número total de shocks distintos por hogar)
shock_wide["n_shocks"] = shock_wide[all_shock_cols].sum(axis=1)

df = df.merge(
    shock_wide[["hhid"] + all_shock_cols + ["n_shocks"]],
    on="hhid", how="left"
)

# Limpiar nombres de columnas de shocks
shock_rename = {
    c: "shock_" + c.lower().replace(" ","_").replace("/","_")[:25]
    for c in all_shock_cols
}
df = df.rename(columns=shock_rename)

print(f"    {len(all_shock_cols)} tipos de shocks + índice n_shocks agregados")
print(f"    Hogares sin ningún shock: {(df['n_shocks']==0).mean()*100:.1f}%")


  [8.5] Integrando shocks económicos (sect12)...
    29 tipos de shocks + índice n_shocks agregados
    Hogares sin ningún shock: 18.3%


#### 8.6 — Encoding ordinal de educación

In [23]:
print("  [8.6] Encoding ordinal de educación...")

edu_map = {
    "0. NONE": 0,
    "11. P1": 1, "12. P2": 1, "13. P3": 1,
    "14. P4": 2, "15. P5": 2, "16. P6": 2,
    "21. JS1": 3, "22. JS2": 3, "23. JS3": 3,
    "24. SS1": 4, "25. SS2": 4, "26. SS3": 4,
    "27. LOWER 6": 4, "28. UPPER 6": 4,
    "31. TEACHER CERTIFICATE": 5,
    "33. MODERN SCHOOL": 5,
    "34. NCE": 5,
    "35. NURSING SCHOOL": 5,
    "41. POLY/PROF": 6,
    "51. QUARANIC": 1,
    "52. ISLAMIC EDUCATION": 1,
    "61. ADULT EDUCATION": 3,
    "62. BASIC LITERACY PRO": 2,
    "64. POST LITERACY 4-11": 3,
    "99. DON'T KNOW": np.nan,
    "411. OND1/ND1 OND2/ND2": 6,
    "412. HND1 HND2": 7,
    "421. UNIVERSITY- LEVEL": 7,
    "422. UNIVERSITY- 400": 7,
    "423. UNIVERSITY- 500": 7,
    "424. UNIVERSITY- 600 L": 7,
    "321. SECONDARY VOCATIO": 5,
    "322. TERTIARY VOCATIO": 6,
    "43. HIGHER DEGREE": 8,
    "2. N2": 0,
}
df["edu_nivel"] = df["s1q28"].map(edu_map)
df = df.drop(columns=["s1q28"])

print(f"    Distribución edu_nivel:")
print(df["edu_nivel"].value_counts(dropna=False).sort_index().to_string())


  [8.6] Encoding ordinal de educación...
    Distribución edu_nivel:
edu_nivel
0.0    4730
1.0    1984
2.0    1525
3.0     107
4.0    1163
5.0     345
6.0      20
8.0     154
NaN    6562


#### 8.7 — Encoding zona geográfica (dummies)

In [24]:
print("  [8.7] Dummies de zona geográfica...")

df["zona"] = df["zone"].str.split(". ").str[-1].str.strip()
zone_dummies = pd.get_dummies(df["zona"], prefix="zona", drop_first=True, dtype=int)
df = pd.concat([df.drop(columns=["zone","zona"]), zone_dummies], axis=1)

print(f"    Zonas como dummies: {list(zone_dummies.columns)}")


  [8.7] Dummies de zona geográfica...
    Zonas como dummies: ['zona_East', 'zona_South', 'zona_West']


#### 8.8 — Imputación de valores faltantes

In [25]:
print("  [8.8] Imputando valores faltantes...")

impute_log = []

# Educación (30% missing) → mediana ordinal
edu_med = df["edu_nivel"].median()
n = df["edu_nivel"].isnull().sum()
df["edu_nivel"] = df["edu_nivel"].fillna(edu_med)
impute_log.append(("edu_nivel", n, f"mediana = {edu_med:.0f}"))

# Remesas (missing = no recibe) → 0
for c in ["remesas_exterior", "remesas_interior"]:
    n = df[c].isnull().sum()
    df[c] = df[c].fillna(0)
    impute_log.append((c, n, "0 (no recibe)"))

# Ingresos (missing = sin empleo formal) → 0
n = df["ingreso_mensual"].isnull().sum()
df["ingreso_mensual"] = df["ingreso_mensual"].fillna(0)
impute_log.append(("ingreso_mensual", n, "0 (sin empleo formal)"))

# Peso muestral (0.8% missing) → media
wt_mean = df["wt_cross_wave5"].mean()
n = df["wt_cross_wave5"].isnull().sum()
df["wt_cross_wave5"] = df["wt_cross_wave5"].fillna(wt_mean)
impute_log.append(("wt_cross_wave5", n, f"media = {wt_mean:.2f}"))

# Shocks: missing = hogar no en la base de shocks → 0
shock_cols_in_df = [c for c in df.columns if c.startswith("shock_")]
for c in shock_cols_in_df + ["n_shocks"]:
    n = df[c].isnull().sum()
    if n > 0:
        df[c] = df[c].fillna(0)

# target_harvest: 11.3% missing → imputar con moda del target planting
th_mode = df["target_harvest"].mode()[0]
n = df["target_harvest"].isnull().sum()
df["target_harvest"] = df["target_harvest"].fillna(th_mode)
impute_log.append(("target_harvest", n, f"moda = {th_mode:.0f} (del target planting)"))

print(f"    {'Variable':<22} {'N imputados':>12}  Criterio")
print(f"    {'-'*55}")
for var, n_imp, crit in impute_log:
    print(f"    {var:<22} {n_imp:>12,}  {crit}")


  [8.8] Imputando valores faltantes...
    Variable                N imputados  Criterio
    -------------------------------------------------------
    edu_nivel                     6,562  mediana = 1
    remesas_exterior                  0  0 (no recibe)
    remesas_interior                  0  0 (no recibe)
    ingreso_mensual              16,590  0 (sin empleo formal)
    wt_cross_wave5                  141  media = 8453.07
    target_harvest                1,871  moda = 1 (del target planting)


    Variable                N imputados  Criterio
    -------------------------------------------------------
    edu_nivel                     6,562  mediana = 1
    remesas_exterior                  0  0 (no recibe)
    remesas_interior                  0  0 (no recibe)
    ingreso_mensual              16,590  0 (sin empleo formal)
    wt_cross_wave5                  141  media = 8453.07
    target_harvest                1,871  moda = 1 (del target planting)


#### 8.9 — Verificación de missings restantes

In [26]:
print("  [8.9] Verificando missings restantes...")

miss_final = df.isnull().mean() * 100
miss_final = miss_final[miss_final > 0].sort_values(ascending=False)

if len(miss_final) == 0:
    print("    ✓ Sin valores faltantes en el dataset final (exceptuando IDs)")
else:
    print(f"    Variables con missings restantes: {len(miss_final)}")
    for v, pct in miss_final.items():
        print(f"      {v:<30} {pct:.1f}%")


  [8.9] Verificando missings restantes...
    Variables con missings restantes: 2
      s4aq4                          0.0%
      s9q2a                          0.0%


#### 8.10 — Dataset definitivo: resumen y guardado

In [27]:
print("  [8.10] Resumen del dataset definitivo...")

# Separar features de IDs/pesos/targets
id_cols      = ["hhid", "indiv"]
weight_col   = ["wt_cross_wave5"]
target_col   = ["target"]
target2_col  = ["target_harvest"]
feature_cols = [c for c in df.columns
                if c not in id_cols + weight_col + target_col + target2_col]

# Dataset para ML (features + target principal)
df_ml    = df[feature_cols + target_col].copy()
# Dataset con pesos y doble target (para análisis de robustez)
df_pesos = df[id_cols + weight_col + target_col + target2_col + feature_cols].copy()

df_ml.to_csv(f"{OUT}/dataset_ml.csv", index=False)
df_pesos.to_csv(f"{OUT}/dataset_ml_con_pesos.csv", index=False)

# ── Gráfico resumen del dataset final ─────────────────
grupos = {
    "Demografía"    : [c for c in feature_cols if c in ["s1q2","s1q6","s1q8","s1q25","edu_nivel"]],
    "Geografía"     : [c for c in feature_cols if c.startswith("zona_")],
    "Remesas"       : [c for c in feature_cols if "remesa" in c],
    "Laboral"       : [c for c in feature_cols if c in ["s4aq4","tiene_empleo","ingreso_mensual"]],
    "Seg. alimentaria": [c for c in feature_cols if c.startswith("s9q")],
    "Shocks"        : [c for c in feature_cols if "shock" in c or c == "n_shocks"],
    "Target harvest": [c for c in feature_cols if c == "target_harvest"],
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Sección 8 — Dataset Definitivo para ML", fontweight="bold")

ax = axes[0]
g_names  = list(grupos.keys())
g_counts = [len(v) for v in grupos.values()]
bars = ax.barh(g_names, g_counts, color=SECONDARY, alpha=0.85, height=0.55)
ax.set_xlabel("Número de features")
ax.set_title(f"Features por grupo temático\n(total: {len(feature_cols)} features + 1 target_harvest)",
             fontweight="bold")
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.invert_yaxis()
for bar, v in zip(bars, g_counts):
    ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
            str(v), va="center", fontsize=10)

ax = axes[1]
counts = df_ml["target"].value_counts().sort_index()
labels = ["No desea migrar (0)", "Desea migrar (1)"]
bars = ax.bar(labels, [counts.get(0,0), counts.get(1,0)],
              color=[ACCENT, SECONDARY], alpha=0.85, width=0.45)
ax.set_ylabel("Observaciones")
ax.set_title("Balance del target\n(clase desbalanceada → usar class_weight)",
             fontweight="bold")
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
for bar in bars:
    pct = bar.get_height() / len(df_ml) * 100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
            f"{bar.get_height():,}\n({pct:.1f}%)",
            ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout()
save(fig, "08_dataset_definitivo")

print("\n" + "="*55)
print("DATASET DEFINITIVO — RESUMEN")
print("="*55)
print(f"  Observaciones:    {len(df_ml):,}")
print(f"  Features totales: {len(feature_cols)}  (+target_harvest como feature adicional)")
for g, cols in grupos.items():
    print(f"    {g:<22} {len(cols):>3} features")
print(f"  Target = 1 (migra): {df_ml['target'].mean()*100:.1f}%")
print(f"  Target = 0 (no):    {(1-df_ml['target'].mean())*100:.1f}%")
print(f"  Missings restantes: {df_ml.isnull().sum().sum()}")
print(f"  Archivos guardados: dataset_ml.csv  |  dataset_ml_con_pesos.csv")
print("="*55)


  [8.10] Resumen del dataset definitivo...
  v 08_dataset_definitivo.png

DATASET DEFINITIVO — RESUMEN
  Observaciones:    16,590
  Features totales: 55  (+target_harvest como feature adicional)
    Demografía               5 features
    Geografía                3 features
    Remesas                  2 features
    Laboral                  3 features
    Seg. alimentaria        12 features
    Shocks                  30 features
    Target harvest           0 features
  Target = 1 (migra): 71.0%
  Target = 0 (no):    29.0%
  Missings restantes: 2
  Archivos guardados: dataset_ml.csv  |  dataset_ml_con_pesos.csv


---
## BONUS — Gráficos Adicionales
Gráficos opcionales para enriquecer la presentación.

In [28]:
# ═══════════════════════════════════════════════════════════════════════
# BONUS 1 — Distribución de edad: migrantes vs. no migrantes
# ═══════════════════════════════════════════════════════════════════════
print("[B1] Distribución de edad por aspiración migratoria...")

# Usar tgt_demo (construido en Sección 3 - requiere correr esa celda antes)
fig, ax = plt.subplots(figsize=(12, 5))

for val, label, color in [
    (1, "Desea migrar",    SECONDARY),
    (0, "No desea migrar", ACCENT),
]:
    subset = tgt_demo[tgt_demo["target"] == val]["s1q6"].dropna()

    # Histograma de fondo (semi-transparente)
    ax.hist(subset, bins=35, density=True, alpha=0.18, color=color)

    # Curva de densidad kernel
    subset.plot.kde(ax=ax, label=f"{label}  (mediana: {subset.median():.0f} años)",
                    color=color, linewidth=2.2)

    # Línea de mediana
    ax.axvline(subset.median(), ls="--", color=color, alpha=0.6, linewidth=1.2)

ax.set_xlabel("Edad (años)")
ax.set_ylabel("Densidad de probabilidad")
ax.set_title(
    "Distribución de edad según aspiración migratoria\n"
    "Nigeria GHS Wave 5 — post-planting 2023/24  |  adultos 15+",
    fontweight="bold"
)
ax.legend(frameon=False, fontsize=10)
ax.set_xlim(14, 90)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)

ax.text(0.98, 0.95,
        "📌 Los adultos (30-50) muestran mayor\n"
        "propensión migratoria → mayor\n"
        "acumulación de capital humano\n"
        "valorable en el exterior",
        transform=ax.transAxes, fontsize=8.5, va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#EBF5FB", alpha=0.8))

plt.tight_layout()
save(fig, "B1_edad_migrantes")
print("  v B1_edad_migrantes.png")


[B1] Distribución de edad por aspiración migratoria...
  v B1_edad_migrantes.png
  v B1_edad_migrantes.png


### Bonus 2 — Distribución HFIAS

In [29]:
# ═══════════════════════════════════════════════════════════════════════
# BONUS 2 — Distribución de inseguridad alimentaria HFIAS
# ═══════════════════════════════════════════════════════════════════════
# NOTA: s9q2a NO es el score HFIAS (es una escala 1-5 de frecuencia).
# El score HFIAS real se calcula como la suma de los 9 ítems s9q1a–s9q1i
# (cada uno YES=1, NO=0), dando un rango 0–9.
print("[B2] Distribución HFIAS...")

hfias_items = ["s9q1a","s9q1b","s9q1c","s9q1d","s9q1e",
               "s9q1f","s9q1g","s9q1h","s9q1i","s9q1j"]

# Etiquetas legibles por ítem (orden del cuestionario GHS Nigeria)
item_labels = {
    "s9q1a": "Preocupación por falta de alimentos",
    "s9q1b": "No pudo comer alimentos preferidos",
    "s9q1c": "Comió solo pocos tipos de alimentos",
    "s9q1d": "Comió alimentos no deseados",
    "s9q1e": "Comió porciones más pequeñas",
    "s9q1f": "Redujo número de comidas",
    "s9q1g": "Se quedó sin alimento en el hogar",
    "s9q1h": "Pasó todo el día sin comer",
    "s9q1i": "Algún miembro se acostó con hambre",
    "s9q1j": "Algún miembro ayunó por falta de alimento",
}

# ── CALCULAR SCORE HFIAS (0–10) ──────────────────────────────────────
food_hfias = food[["hhid"] + hfias_items].copy()
for col in hfias_items:
    food_hfias[col] = food_hfias[col].map({"1. YES": 1, "2. NO": 0})

food_hfias["hfias_score"] = food_hfias[hfias_items].sum(axis=1)

# Categorías usando umbrales para escala 0–10
# Basado en proporción equivalente al estándar 0–27:
# 0 → Seguro | 1–3 → Lev. inseguro | 4–7 → Mod. inseguro | 8–10 → Sev. inseguro
cat_labels = ["Seguro (0)", "Levemente inseguro (1-3)", "Moderadamente inseguro (4-7)", "Severamente inseguro (8-10)"]
food_hfias["hfias_cat"] = pd.cut(
    food_hfias["hfias_score"],
    bins=[-0.5, 0.5, 3.5, 7.5, 10.5],
    labels=cat_labels
)

cat_pcts   = food_hfias["hfias_cat"].value_counts()[cat_labels]
cat_pcts   = cat_pcts / cat_pcts.sum() * 100
cat_counts = food_hfias["hfias_cat"].value_counts()[cat_labels]

# % YES por ítem (para panel izq)
item_pct = {col: food_hfias[col].mean() * 100 for col in hfias_items}
item_df   = pd.Series(item_pct).rename(index=item_labels).sort_values()

# ── FIGURA ─────────────────────────────────────────────────────────────
colors_hfias = ["#27AE60", "#F1C40F", "#E67E22", "#E74C3C"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "Bonus 2 — Inseguridad Alimentaria (HFIAS)\n"
    "Nigeria GHS Wave 5 — post-planting 2023/24",
    fontweight="bold"
)

# Panel izq: % de hogares que experimentó cada condición alimentaria
ax = axes[0]
bar_colors_items = [ACCENT if v >= 50 else "#F39C12" if v >= 25 else SECONDARY
                    for v in item_df.values]
bars = ax.barh(item_df.index, item_df.values,
               color=bar_colors_items, alpha=0.85, height=0.6)
ax.axvline(50, ls="--", color=GRAY, linewidth=1, alpha=0.7, label="50%")
ax.set_xlabel("% de hogares afectados")
ax.set_title("% de hogares que experimentó cada\ncondición de inseguridad alimentaria",
             fontweight="bold")
ax.set_xlim(0, 100)
ax.legend(fontsize=8, frameon=False)
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False, labelsize=8.5)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 1, bar.get_y() + bar.get_height() / 2,
            f"{w:.1f}%", va="center", ha="left", fontsize=8)

# Panel der: distribución por categoría HFIAS
ax2 = axes[1]
bars2 = ax2.bar(cat_labels, cat_pcts.values,
                color=colors_hfias, alpha=0.88, width=0.55, edgecolor="white")
for bar, pct, n in zip(bars2, cat_pcts.values, cat_counts.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f"{pct:.1f}%\n(n={n:,})",
             ha="center", va="bottom", fontsize=10, fontweight="bold")
ax2.set_ylabel("% de hogares")
ax2.set_title("Distribución por categoría HFIAS\n"
              "(score = suma de ítems afectados, escala 0–10)",
              fontweight="bold")
ax2.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax2.grid(axis="x", visible=False)
ax2.set_ylim(0, cat_pcts.max() * 1.2)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.tick_params(axis="x", labelsize=8.5)

# Anotar score promedio
mean_score = food_hfias["hfias_score"].mean()
ax2.text(0.97, 0.97, f"Score promedio: {mean_score:.1f} / 10",
         transform=ax2.transAxes, ha="right", va="top", fontsize=9,
         bbox=dict(boxstyle="round,pad=0.3", facecolor="#EBF5FB", alpha=0.9))

plt.tight_layout()
save(fig, "B2_hfias_distribucion")
print("  v B2_hfias_distribucion.png")


[B2] Distribución HFIAS...
  v B2_hfias_distribucion.png
  v B2_hfias_distribucion.png


### Bonus 3 — Distribución geográfica (mapa)

In [30]:
# ═══════════════════════════════════════════════════════════════════════
# BONUS 3 — Mapa estilizado de Nigeria: encuestados por zona geográfica
# ═══════════════════════════════════════════════════════════════════════
print("[B3] Distribución geográfica por zona de Nigeria...")

# CORRECCIÓN: split en "." (no ". " — el punto es wildcard regex)
zone_hh = demo.groupby(["hhid", "zone"]).size().reset_index(name="_cnt")
zone_hh = zone_hh.drop_duplicates("hhid")
zone_counts = zone_hh.groupby("zone")["_cnt"].count().reset_index()
zone_counts.columns = ["zone_raw", "n_hogares"]
zone_counts["zone_clean"] = (zone_counts["zone_raw"]
                             .str.split(".")
                             .str[-1]
                             .str.strip()
                             .str.title())
zone_counts = zone_counts.sort_values("n_hogares", ascending=False).reset_index(drop=True)
print("Zonas:", zone_counts["zone_clean"].tolist())

# Coordenadas reales aproximadas del centro de cada zona geopolítica
zone_info = {
    "North West"   : {"lat": 12.0, "lon": 7.0,  "color": "#1ABC9C"},
    "North East"   : {"lat": 11.0, "lon": 12.5, "color": "#2980B9"},
    "North Central": {"lat":  9.0, "lon": 8.5,  "color": PALETTE},
    "South West"   : {"lat":  7.5, "lon": 4.5,  "color": ACCENT},
    "South South"  : {"lat":  5.2, "lon": 6.5,  "color": "#8E44AD"},
    "South East"   : {"lat":  6.0, "lon": 8.0,  "color": SECONDARY},
}

# Contorno de Nigeria con ~30 puntos (mucho más preciso)
# Sentido antihorario desde el extremo SO (Lagos)
nigeria_pts = [
    # Costa sur / delta del Niger
    (3.3,  6.4),   # Lagos
    (2.7,  6.3),   # frontera con Benin (SO)
    # Frontera oeste, subiendo hacia el norte
    (2.7,  7.0),
    (2.8,  8.0),
    (3.0,  9.5),
    (3.2, 11.0),
    (3.6, 11.5),
    (4.0, 12.5),
    (4.5, 13.0),
    (5.0, 13.5),
    # Frontera norte (aproximadamente recta)
    (6.0, 13.8),
    (7.5, 13.9),
    (9.0, 13.7),
    (10.5, 13.5),
    (12.0, 13.3),
    (13.5, 13.5),
    (14.2, 13.1),
    (14.6, 12.5),  # vértice NE
    # Frontera este (Camerún), bajando
    (14.5, 11.5),
    (14.2, 10.0),
    (13.5,  8.5),
    (13.2,  7.0),
    (12.5,  6.0),
    (11.0,  5.5),
    # Costa sur / delta del Niger (de E a O)
    ( 9.5,  4.5),
    ( 8.5,  4.3),
    ( 7.5,  4.3),
    ( 6.5,  4.3),
    ( 5.5,  4.9),
    ( 4.8,  5.8),
    ( 4.0,  6.2),
    ( 3.3,  6.4),  # cierre en Lagos
]

import matplotlib.patches as mpatches
from matplotlib.path import Path

codes = [Path.MOVETO] + [Path.LINETO] * (len(nigeria_pts) - 2) + [Path.CLOSEPOLY]

fig = plt.figure(figsize=(16, 6.5))
fig.suptitle(
    "Bonus 3 — Distribución Geográfica de Encuestados\n"
    "Nigeria GHS Wave 5 — post-planting 2023/24",
    fontweight="bold", y=1.01
)

# ── Panel izq: mapa ─────────────────────────────────────────────────────
ax_map = fig.add_axes([0.02, 0.05, 0.44, 0.90])
ax_map.set_facecolor("#EAF2F8")
ax_map.set_xlim(1.8, 15.8)
ax_map.set_ylim(3.5, 14.5)
ax_map.set_aspect("equal")
ax_map.set_title("Zonas geopoliticas de Nigeria\n(tamano de burbuja ~ hogares encuestados)",
                 fontweight="bold", pad=8)
ax_map.set_xlabel("Longitud")
ax_map.set_ylabel("Latitud")
ax_map.grid(color="#D5E8F0", linewidth=0.5, zorder=0)
ax_map.spines["top"].set_visible(False)
ax_map.spines["right"].set_visible(False)

patch = mpatches.PathPatch(Path(nigeria_pts, codes),
                           facecolor="#F0F3F4", edgecolor="#2C3E50",
                           linewidth=1.8, alpha=0.95, zorder=1)
ax_map.add_patch(patch)

max_n = zone_counts["n_hogares"].max()
for _, row in zone_counts.iterrows():
    zname = row["zone_clean"]
    info  = zone_info.get(zname)
    if info is None:
        continue
    lon, lat = info["lon"], info["lat"]
    n        = row["n_hogares"]
    size     = (n / max_n) * 2000 + 400
    color    = info["color"]

    ax_map.scatter(lon, lat, s=size, color=color, alpha=0.80,
                   edgecolors="white", linewidths=1.5, zorder=3)
    ax_map.text(lon, lat + 0.60, zname.replace(" ", "\n"),
                ha="center", va="bottom", fontsize=7.5,
                fontweight="bold", color="#2C3E50", zorder=4)
    ax_map.text(lon, lat, f"n={n:,}",
                ha="center", va="center", fontsize=7,
                color="white", fontweight="bold", zorder=5)

ax_map.text(0.02, 0.01,
            "Nota: mapa esquematico — contorno simplificado",
            transform=ax_map.transAxes, fontsize=7, color=GRAY, va="bottom")

# ── Panel der: barras ───────────────────────────────────────────────────
ax_bar = fig.add_axes([0.55, 0.08, 0.42, 0.82])

bar_colors = [zone_info.get(z, {}).get("color", GRAY) for z in zone_counts["zone_clean"]]
bars = ax_bar.barh(zone_counts["zone_clean"], zone_counts["n_hogares"],
                   color=bar_colors, alpha=0.88, height=0.6)
ax_bar.invert_yaxis()
ax_bar.set_xlabel("Numero de hogares encuestados")
ax_bar.set_title("Hogares encuestados por zona", fontweight="bold")
ax_bar.set_xlim(0, zone_counts["n_hogares"].max() * 1.28)
ax_bar.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax_bar.grid(axis="y", visible=False)
ax_bar.spines["left"].set_visible(False)
ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)
ax_bar.tick_params(left=False, labelsize=9)

total_hh = zone_counts["n_hogares"].sum()
for bar, (_, row) in zip(bars, zone_counts.iterrows()):
    pct  = row["n_hogares"] / total_hh * 100
    xpos = bar.get_width() + zone_counts["n_hogares"].max() * 0.02
    ax_bar.text(xpos, bar.get_y() + bar.get_height() / 2,
                f"{row['n_hogares']:,}  ({pct:.1f}%)",
                va="center", ha="left", fontsize=9)

save(fig, "B3_mapa_nigeria")
print("  v B3_mapa_nigeria.png")

[B3] Distribución geográfica por zona de Nigeria...
Zonas: ['North Central', 'North East', 'South East', 'South South', 'North West', 'South West']
  v B3_mapa_nigeria.png
  v B3_mapa_nigeria.png


### ANALISIS DE MODELOS (ver cual tiene mayor performance)

### SetUp: Librerías y carga de DataBases

In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection   import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing     import StandardScaler
from sklearn.pipeline          import Pipeline
from sklearn.impute            import SimpleImputer
from sklearn.linear_model      import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble          import RandomForestClassifier
from sklearn.metrics           import (
    roc_auc_score, roc_curve,
    classification_report, confusion_matrix,
    f1_score, balanced_accuracy_score
)

# ── Cargar dataset limpio ─────────────────────────────────────────────
df_ml = pd.read_csv(f"{OUT}/dataset_ml.csv")

# Eliminar las 2 filas con NaN restantes (de 16,590 — impacto estadístico nulo)
n_antes = len(df_ml)
df_ml   = df_ml.dropna()
print(f"Filas antes de dropna: {n_antes:,}")
print(f"Filas después:         {len(df_ml):,}  (eliminadas: {n_antes - len(df_ml)})")
print(f"Missings restantes:    {df_ml.isnull().sum().sum()}")

# ── Feature engineering final ─────────────────────────────────────────
# Dummy que indica si el individuo tiene datos laborales completos.
# Los ingresos/horas tienen ~49% missing imputado a 0 en la sección 8;
# este dummy captura si el PATRÓN de missing en sí mismo predice la aspiración.
df_ml['tiene_datos_laborales'] = (df_ml['ingreso_mensual'] != 0).astype(int)
print(f"\nFeature 'tiene_datos_laborales' agregada")
print(f"  Con datos laborales:    {df_ml['tiene_datos_laborales'].sum():,} "
      f"({df_ml['tiene_datos_laborales'].mean()*100:.1f}%)")
print(f"  Sin datos laborales:    {(df_ml['tiene_datos_laborales']==0).sum():,} "
      f"({(df_ml['tiene_datos_laborales']==0).mean()*100:.1f}%)")

# ── Separar X e y ────────────────────────────────────────────────────
y             = df_ml['target'].values
X             = df_ml.drop(columns=['target']).values
feature_names = df_ml.drop(columns=['target']).columns.tolist()

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}  |  positivos: {y.sum():,} ({y.mean()*100:.1f}%)")
print(f"Features: {len(feature_names)}")

Filas antes de dropna: 16,590
Filas después:         16,588  (eliminadas: 2)
Missings restantes:    0

Feature 'tiene_datos_laborales' agregada
  Con datos laborales:    0 (0.0%)
  Sin datos laborales:    16,588 (100.0%)

X shape: (16588, 56)
y shape: (16588,)  |  positivos: 11,781 (71.0%)
Features: 56


### Train/Test 

In [32]:
# Estratificado: garantiza que train y test mantengan el mismo balance 71/29
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.30, #coni
    random_state = 42,
    stratify     = y
)

print(f"Train: {X_train.shape[0]:,} obs  |  positivos: {y_train.mean()*100:.1f}%")
print(f"Test:  {X_test.shape[0]:,}  obs  |  positivos: {y_test.mean()*100:.1f}%")

# Esquema de CV: se aplica SOLO sobre train (test nunca se toca durante el ajuste)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\nEsquema: 5-fold StratifiedKFold sobre train")

Train: 11,611 obs  |  positivos: 71.0%
Test:  4,977  obs  |  positivos: 71.0%

Esquema: 5-fold StratifiedKFold sobre train


### Modelo A: Logistic regression

In [33]:
print("[Modelo A] Logistic Regression (baseline)...")

# Pipeline: imputar (por si quedara algún NaN) → estandarizar → logit
# class_weight='balanced' ajusta el peso de la clase minoritaria (NO migra, 29%)
# para que el modelo no la ignore durante el entrenamiento
pipe_logit = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(
                    penalty      = None,        # sin regularización
                    class_weight = 'balanced',
                    max_iter     = 1000,
                    solver       = 'lbfgs',
                    random_state = 42
                ))
])

# Cross-validation sobre train: estima performance sin tocar el test set
cv_logit = cross_validate(
    pipe_logit, X_train, y_train,
    cv                 = cv,
    scoring            = ['roc_auc', 'f1_weighted', 'balanced_accuracy'],
    return_train_score = False
)

# Entrenamos sobre TODO el train para evaluar en test
pipe_logit.fit(X_train, y_train)
y_prob_logit = pipe_logit.predict_proba(X_test)[:, 1]
y_pred_logit = pipe_logit.predict(X_test)

auc_logit  = roc_auc_score(y_test, y_prob_logit)
f1_logit   = f1_score(y_test, y_pred_logit, average='weighted')
bacc_logit = balanced_accuracy_score(y_test, y_pred_logit)

print(f"  AUC-ROC  (test):        {auc_logit:.4f}")
print(f"  Balanced Accuracy:      {bacc_logit:.4f}")
print(f"  F1 weighted (test):     {f1_logit:.4f}")
print(f"  AUC-ROC  (CV mean):     {cv_logit['test_roc_auc'].mean():.4f} "
      f"± {cv_logit['test_roc_auc'].std():.4f}")
print("  OK")

[Modelo A] Logistic Regression (baseline)...
  AUC-ROC  (test):        0.6679
  Balanced Accuracy:      0.6171
  F1 weighted (test):     0.6347
  AUC-ROC  (CV mean):     0.6831 ± 0.0122
  OK


### Modelo B: Elastic Net (logística regularizada)

**Intuición:** Es una logística con dos tipos de penalización simultáneas:
- **L1 (LASSO):** empuja coeficientes exactamente a cero → selecciona automáticamente cuáles variables importan. Muy útil con los 30 shocks que son mayormente ceros.
- **L2 (Ridge):** encoge todos los coeficientes proporcionalmente → estabiliza el modelo cuando dos variables están correlacionadas (ej: varios shocks climáticos juntos).

El parámetro `l1_ratio` controla el mix: 1.0 = LASSO puro, 0.0 = Ridge puro. `LogisticRegressionCV` elige automáticamente la penalización óptima (C) por cross-validation.

In [34]:
print("[Modelo B] Elastic Net (logística regularizada)...")
print("  (puede tardar 2-3 minutos — explora grilla de C x l1_ratio con CV)")

# Primero imputamos y escalamos X_train/X_test manualmente
# porque LogisticRegressionCV no acepta Pipeline directamente con CV interno
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

# LogisticRegressionCV busca el mejor C (inverso de regularización) y l1_ratio
# usando el mismo CV definido arriba, optimizando AUC-ROC
enet_cv = LogisticRegressionCV(
    penalty      = 'elasticnet',
    solver       = 'saga',
    l1_ratios    = [0.5, 0.7, 0.9, 1.0],   # grilla: de mix 50/50 hasta LASSO puro
    Cs           = 10,                       # 10 valores de C en escala log
    cv           = cv,
    scoring      = 'roc_auc',
    class_weight = 'balanced',
    max_iter     = 3000,
    random_state = 42,
    n_jobs       = -1
)
enet_cv.fit(X_train_prep, y_train)

# Parámetros elegidos por CV
best_C    = enet_cv.C_[0]
best_l1   = enet_cv.l1_ratio_[0]
coefs     = enet_cv.coef_[0]
n_nonzero = np.sum(np.abs(coefs) > 1e-6)

y_prob_enet = enet_cv.predict_proba(X_test_prep)[:, 1]
y_pred_enet = enet_cv.predict(X_test_prep)

auc_enet  = roc_auc_score(y_test, y_prob_enet)
f1_enet   = f1_score(y_test, y_pred_enet, average='weighted')
bacc_enet = balanced_accuracy_score(y_test, y_pred_enet)

print(f"  C óptimo:                     {best_C:.4f}")
print(f"  l1_ratio óptimo:              {best_l1:.2f}  (1.0 = LASSO puro)")
print(f"  Variables activas (coef≠0):   {n_nonzero} de {len(coefs)}")
print(f"  Variables zeroed out:         {len(coefs) - n_nonzero}")
print(f"  AUC-ROC  (test):              {auc_enet:.4f}")
print(f"  Balanced Accuracy:            {bacc_enet:.4f}")
print(f"  F1 weighted (test):           {f1_enet:.4f}")
print("  OK")

[Modelo B] Elastic Net (logística regularizada)...
  (puede tardar 2-3 minutos — explora grilla de C x l1_ratio con CV)
  C óptimo:                     0.0464
  l1_ratio óptimo:              1.00  (1.0 = LASSO puro)
  Variables activas (coef≠0):   47 de 56
  Variables zeroed out:         9
  AUC-ROC  (test):              0.6694
  Balanced Accuracy:            0.6211
  F1 weighted (test):           0.6376
  OK


### 9.4 — Modelo C: Random Forest


**Intuición:** Construye cientos de árboles de decisión, cada uno entrenado sobre una muestra aleatoria de los datos y usando un subconjunto aleatorio de variables en cada split. El resultado final es el promedio de todos los árboles. La aleatoriedad doble hace que los árboles sean distintos entre sí y el promedio sea más robusto que cualquier árbol individual.

Ventajas clave para este dataset: no necesita escalar variables, maneja bien las 30+ dummies de shocks, captura interacciones (ej: *joven + sin trabajo + shock climático*), y entrega directamente qué variables fueron más usadas para hacer splits → **variable importance**.

In [35]:
print("[Modelo C] Random Forest...")

# RF no necesita scaler (los árboles son invariantes a escala)
# pero sí necesita que no haya NaN — usamos X_train_prep que ya está imputado
# (el scaler no afecta a RF, pero el imputer sí es necesario)
rf = RandomForestClassifier(
    n_estimators     = 300,     # 300 árboles: buena estabilidad sin exceso de costo
    max_depth        = None,    # árboles completos
    min_samples_leaf = 20,      # hoja mínima de 20 obs: evita memorizar ruido
    max_features     = 'sqrt',  # raíz cuadrada de features por split (estándar)
    class_weight     = 'balanced',
    oob_score        = True,    # estimación out-of-bag gratuita (aproxima CV)
    random_state     = 42,
    n_jobs           = -1
)

# CV sobre train (usamos X_train_prep que ya está imputado)
cv_rf = cross_validate(
    rf, X_train_prep, y_train,
    cv                 = cv,
    scoring            = ['roc_auc', 'f1_weighted', 'balanced_accuracy'],
    return_train_score = False
)

# Entrenamos sobre todo el train
rf.fit(X_train_prep, y_train)

y_prob_rf = rf.predict_proba(X_test_prep)[:, 1]
y_pred_rf = rf.predict(X_test_prep)

auc_rf  = roc_auc_score(y_test, y_prob_rf)
f1_rf   = f1_score(y_test, y_pred_rf, average='weighted')
bacc_rf = balanced_accuracy_score(y_test, y_pred_rf)

print(f"  OOB Score (aprox. CV):  {rf.oob_score_:.4f}")
print(f"  AUC-ROC  (test):        {auc_rf:.4f}")
print(f"  Balanced Accuracy:      {bacc_rf:.4f}")
print(f"  F1 weighted (test):     {f1_rf:.4f}")
print(f"  AUC-ROC  (CV mean):     {cv_rf['test_roc_auc'].mean():.4f} "
      f"± {cv_rf['test_roc_auc'].std():.4f}")
print("  OK")

[Modelo C] Random Forest...
  OOB Score (aprox. CV):  0.6773
  AUC-ROC  (test):        0.7070
  Balanced Accuracy:      0.6509
  F1 weighted (test):     0.6861
  AUC-ROC  (CV mean):     0.7138 ± 0.0129
  OK


### Comparación de Modelos

In [36]:
print("[9.5] Gráfico de comparación de modelos...")

os.makedirs(OUT, exist_ok=True)

# ── Paleta (redefinida aquí por si se corre la sección 9 de forma aislada) ──
PALETTE   = "#2C3E50"
ACCENT    = "#E74C3C"
SECONDARY = "#3498DB"
GRAY      = "#95A5A6"

COL_LOGIT = PALETTE
COL_ENET  = SECONDARY
COL_RF    = ACCENT

def save(fig, name):
    path = f"{OUT}/{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  v {name}.png")

# Reportes completos por clase (necesarios para la tabla)
rep_logit = classification_report(y_test, y_pred_logit, output_dict=True)
rep_enet  = classification_report(y_test, y_pred_enet,  output_dict=True)
rep_rf    = classification_report(y_test, y_pred_rf,    output_dict=True)

# ── Curvas ROC ────────────────────────────────────────────────────────
fpr_logit, tpr_logit, _ = roc_curve(y_test, y_prob_logit)
fpr_enet,  tpr_enet,  _ = roc_curve(y_test, y_prob_enet)
fpr_rf,    tpr_rf,    _ = roc_curve(y_test, y_prob_rf)

fig = plt.figure(figsize=(18, 10))
fig.suptitle(
    "Sección 9 — Comparación de Modelos Predictivos\n"
    "Nigeria GHS Wave 5 — Aspiraciones Migratorias",
    fontweight="bold", y=1.01
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Panel 1: Curvas ROC superpuestas ─────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(fpr_logit, tpr_logit, color=COL_LOGIT, lw=2,
         label=f"Logit baseline  (AUC={auc_logit:.3f})")
ax1.plot(fpr_enet,  tpr_enet,  color=COL_ENET,  lw=2, ls='--',
         label=f"Elastic Net     (AUC={auc_enet:.3f})")
ax1.plot(fpr_rf,    tpr_rf,    color=COL_RF,    lw=2, ls='-.',
         label=f"Random Forest   (AUC={auc_rf:.3f})")
ax1.plot([0,1],[0,1], 'k--', lw=0.8, alpha=0.5, label='Clasificador aleatorio')
ax1.set_xlabel("Tasa de Falsos Positivos")
ax1.set_ylabel("Tasa de Verdaderos Positivos")
ax1.set_title("Curvas ROC (test set)", fontweight="bold")
ax1.legend(fontsize=8, frameon=False)
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1.02)
ax1.grid(color="#E8E8E8", linewidth=0.6)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# ── Panel 2: AUC-ROC por CV (barras con error estándar) ───────────────
ax2 = fig.add_subplot(gs[0, 1])
modelos    = ['Logit\nbaseline', 'Elastic\nNet', 'Random\nForest']
aucs_test  = [auc_logit, auc_enet, auc_rf]
aucs_cv    = [
    cv_logit['test_roc_auc'].mean(),
    auc_enet,   # ElasticNetCV optimiza con CV interno; reportamos directamente el test
    cv_rf['test_roc_auc'].mean()
]
stds_cv    = [
    cv_logit['test_roc_auc'].std(),
    0,
    cv_rf['test_roc_auc'].std()
]
colors_bar = [COL_LOGIT, COL_ENET, COL_RF]
bars = ax2.bar(modelos, aucs_cv, yerr=stds_cv, color=colors_bar,
               alpha=0.85, width=0.45, capsize=5,
               error_kw={'linewidth': 1.5, 'ecolor': '#555555'})
ax2.set_ylabel("AUC-ROC")
ax2.set_title("AUC-ROC promedio\n(5-fold CV sobre train)", fontweight="bold")
ax2.set_ylim(0.5, min(1.0, max(aucs_cv) + 0.10))
ax2.axhline(0.5, ls='--', color='gray', lw=0.8, label='Clasificador aleatorio')
ax2.legend(fontsize=8, frameon=False)
ax2.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax2.grid(axis="x", visible=False)
ax2.spines["left"].set_visible(False)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
for bar, val, std in zip(bars, aucs_cv, stds_cv):
    label = f"{val:.3f}" if std == 0 else f"{val:.3f}\n±{std:.3f}"
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
             label, ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Panel 3: Tabla de métricas ────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.axis('off')
tabla_data = [
    ['Métrica',             'Logit',
                            'Elastic Net',
                            'Rand. Forest'],
    ['AUC-ROC (test)',      f"{auc_logit:.3f}",
                            f"{auc_enet:.3f}",
                            f"{auc_rf:.3f}"],
    ['Balanced Acc.',       f"{bacc_logit:.3f}",
                            f"{bacc_enet:.3f}",
                            f"{bacc_rf:.3f}"],
    ['F1 clase 0\n(no migra)', f"{rep_logit['0']['f1-score']:.3f}",
                            f"{rep_enet['0']['f1-score']:.3f}",
                            f"{rep_rf['0']['f1-score']:.3f}"],
    ['F1 clase 1\n(migra)', f"{rep_logit['1']['f1-score']:.3f}",
                            f"{rep_enet['1']['f1-score']:.3f}",
                            f"{rep_rf['1']['f1-score']:.3f}"],
    ['Precisión cl.0',      f"{rep_logit['0']['precision']:.3f}",
                            f"{rep_enet['0']['precision']:.3f}",
                            f"{rep_rf['0']['precision']:.3f}"],
    ['Recall cl.0',         f"{rep_logit['0']['recall']:.3f}",
                            f"{rep_enet['0']['recall']:.3f}",
                            f"{rep_rf['0']['recall']:.3f}"],
]
tbl = ax3.table(cellText=tabla_data[1:], colLabels=tabla_data[0],
                loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1.1, 1.75)
for j in range(4):
    tbl[0, j].set_facecolor(PALETTE)
    tbl[0, j].set_text_props(color='white', fontweight='bold')
# Highlight la celda con mejor valor en cada fila
for i in range(1, len(tabla_data)):
    vals = []
    for j in range(1, 4):
        try:
            vals.append(float(tabla_data[i][j]))
        except:
            vals.append(-1)
    best_j = int(np.argmax(vals)) + 1
    tbl[i, best_j].set_facecolor('#D5F5E3')
    tbl[i, best_j].set_text_props(fontweight='bold')
ax3.set_title("Métricas comparativas (test set)\nverde = mejor en cada fila",
              fontweight="bold", pad=60)

# ── Panel 4: Coeficientes Elastic Net (top 15 por magnitud) ──────────
ax4 = fig.add_subplot(gs[1, 0:2])
coef_series  = pd.Series(enet_cv.coef_[0], index=feature_names)
coef_nonzero = coef_series[np.abs(coef_series) > 1e-6].sort_values()
top_n        = min(15, len(coef_nonzero))   # por si hay menos de 15 activas
top_coefs    = coef_nonzero.reindex(
    coef_nonzero.abs().nlargest(top_n).index
).sort_values()

colors_coef = [COL_ENET if v >= 0 else COL_RF for v in top_coefs.values]
ax4.barh(top_coefs.index, top_coefs.values,
         color=colors_coef, alpha=0.85, height=0.65)
ax4.axvline(0, color='black', linewidth=0.8)
ax4.set_xlabel("Coeficiente (escala estandarizada)")
ax4.set_title(
    f"Top {top_n} coeficientes — Elastic Net"
    f"  (l1_ratio={best_l1:.2f}, C={best_C:.4f}  |  "
    f"{n_nonzero} variables activas de {len(coef_series)})\n"
    "Azul = aumenta prob. de migrar  |  Rojo = la reduce",
    fontweight="bold"
)
ax4.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax4.grid(axis="y", visible=False)
ax4.spines["left"].set_visible(False)
ax4.spines["top"].set_visible(False)
ax4.spines["right"].set_visible(False)
ax4.tick_params(axis="y", labelsize=8.5)

# ── Panel 5: Feature importance Random Forest (top 15) ───────────────
ax5 = fig.add_subplot(gs[1, 2])
importances = pd.Series(rf.feature_importances_, index=feature_names)
top_imp     = importances.nlargest(15).sort_values()

bars5 = ax5.barh(top_imp.index, top_imp.values,
                 color=COL_RF, alpha=0.85, height=0.65)
ax5.set_xlabel("Importancia (mean decrease impurity)")
ax5.set_title(
    "Top 15 variables — Random Forest\n"
    "(mayor valor = más usada para splits en los 300 árboles)",
    fontweight="bold"
)
ax5.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax5.grid(axis="y", visible=False)
ax5.spines["left"].set_visible(False)
ax5.spines["top"].set_visible(False)
ax5.spines["right"].set_visible(False)
ax5.tick_params(axis="y", labelsize=8.5)
for bar, val in zip(bars5, top_imp.values):
    ax5.text(bar.get_width()+0.0005, bar.get_y()+bar.get_height()/2,
             f"{val:.3f}", va='center', fontsize=7.5)

save(fig, "09_comparacion_modelos")
print("  v 09_comparacion_modelos.png")

[9.5] Gráfico de comparación de modelos...
  v 09_comparacion_modelos.png
  v 09_comparacion_modelos.png


### Matrices de Confusión y resumen

In [37]:
print("[9.6] Matrices de confusión...")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle(
    "Sección 9 — Matrices de Confusión (test set)\n"
    "Filas = clase real  |  Columnas = clase predicha",
    fontweight="bold"
)

for ax, (y_pred, nombre, color) in zip(axes, [
    (y_pred_logit, 'Logit baseline', COL_LOGIT),
    (y_pred_enet,  'Elastic Net',    COL_ENET),
    (y_pred_rf,    'Random Forest',  COL_RF),
]):
    cm     = confusion_matrix(y_test, y_pred)
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred:\nNo migra', 'Pred:\nMigra'], fontsize=9)
    ax.set_yticklabels(['Real:\nNo migra', 'Real:\nMigra'], fontsize=9)
    ax.set_title(nombre, fontweight='bold', color=color, pad=10)
    for i in range(2):
        for j in range(2):
            ax.text(j, i,
                    f"{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)",
                    ha='center', va='center', fontsize=11, fontweight='bold',
                    color='white' if cm_pct[i,j] > 55 else '#2C3E50')

plt.tight_layout()
save(fig, "09b_confusion_matrices")
print("  v 09b_confusion_matrices.png")

# ── Resumen final en consola ──────────────────────────────────────────
print("\n" + "="*65)
print("RESUMEN COMPARATIVO — MODELOS PREDICTIVOS")
print("="*65)
print(f"{'Modelo':<22} {'AUC-ROC':>8} {'Bal.Acc':>8} {'F1(cl.0)':>9} {'F1(cl.1)':>9}")
print("-"*65)
for nombre, auc, bacc, rep in [
    ('Logit baseline',   auc_logit, bacc_logit, rep_logit),
    ('Elastic Net',      auc_enet,  bacc_enet,  rep_enet),
    ('Random Forest',    auc_rf,    bacc_rf,    rep_rf),
]:
    f0 = rep['0']['f1-score']
    f1 = rep['1']['f1-score']
    print(f"{nombre:<22} {auc:>8.4f} {bacc:>8.4f} {f0:>9.3f} {f1:>9.3f}")
print("="*65)
print(f"  Elastic Net: {n_nonzero} variables activas de {len(coef_series)} totales")
print(f"  RF OOB score: {rf.oob_score_:.4f}")
print("")
print("NOTA: con señal débil (r_max~0.20 con target), un AUC de 0.60-0.70")
print("es un resultado sustantivo, no un fracaso. La edad y las variables")
print("demográficas concentran el poder predictivo disponible.")

[9.6] Matrices de confusión...
  v 09b_confusion_matrices.png
  v 09b_confusion_matrices.png

RESUMEN COMPARATIVO — MODELOS PREDICTIVOS
Modelo                  AUC-ROC  Bal.Acc  F1(cl.0)  F1(cl.1)
-----------------------------------------------------------------
Logit baseline           0.6679   0.6171     0.483     0.697
Elastic Net              0.6694   0.6211     0.487     0.699
Random Forest            0.7070   0.6509     0.514     0.756
  Elastic Net: 47 variables activas de 56 totales
  RF OOB score: 0.6773

NOTA: con señal débil (r_max~0.20 con target), un AUC de 0.60-0.70
es un resultado sustantivo, no un fracaso. La edad y las variables
demográficas concentran el poder predictivo disponible.


### Conclusión: cuál es el mejor modelo?

Clasificación binaria con validación cruzada: tres modelos de complejidad creciente (Logit → Elastic Net → Random Forest), seleccionando el ***Random Forest como modelo final por tener el mejor AUC out-of-sample (0.707)***.

---# 🎯 INSTANCIA 2 — Modificaciones para el pósterA partir de acá se agregan las modificaciones pedidas en el PDF de actualizaciones:1. **Heatmap de correlación** entre features (no sólo con el target)2. **Reducción de dimensionalidad** (PCA) — análisis exploratorio3. **Modelo no lineal adicional**: Gradient Boosting (captura no linealidades como RF, pero con boosting secuencial)4. **Curva Precision-Recall** + **distribution plot** de probabilidades predichas5. **Permutation importance** (más robusta que mean decrease impurity)6. **Calibración** del modelo final7. **Matriz de confusión final** detallada del modelo ganador8. **Discusión y limitaciones**Todos los gráficos y tablas que entren al póster se guardan en `graficos_y_tablas_poster/`.

### SECCIÓN 10 — Configurar carpeta de salida del póster

In [38]:
import os, shutil
import matplotlib.pyplot as plt

# Carpeta de salida exclusiva para los gráficos/tablas que irán al póster final.
# Por defecto se crea junto al notebook; si querés cambiar la ubicación, editar acá.
POSTER_OUT = os.path.join(os.path.dirname(OUT), "graficos_y_tablas_poster") \
    if os.path.basename(OUT) == "diagnostico" else "graficos_y_tablas_poster"

os.makedirs(POSTER_OUT, exist_ok=True)
print(f"Carpeta póster: {POSTER_OUT}")

def save_poster(fig, name, dpi=200, also_pdf=True):
    """Guarda figura en alta resolución en POSTER_OUT (PNG y opcionalmente PDF vectorial)."""
    png_path = os.path.join(POSTER_OUT, f"{name}.png")
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    if also_pdf:
        pdf_path = os.path.join(POSTER_OUT, f"{name}.pdf")
        fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  ✓ {name}.png" + ("  +  .pdf" if also_pdf else ""))

# Reutilizamos la paleta del notebook (por si se corre la sección aislada)
PALETTE   = "#2C3E50"
ACCENT    = "#E74C3C"
SECONDARY = "#3498DB"
GRAY      = "#95A5A6"
SUCCESS   = "#27AE60"

COL_LOGIT = PALETTE
COL_ENET  = SECONDARY
COL_RF    = ACCENT
COL_GB    = SUCCESS  # color para el nuevo modelo (Gradient Boosting)


Carpeta póster: /Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/Proyecto_poster/graficos_y_tablas_poster


### SECCIÓN 11 — Heatmap de correlación entre featuresCalculamos la matriz de correlación de Pearson entre **todas las features del modelo final** (las 56 que entran al RF). El objetivo es identificar:- Bloques de variables altamente correlacionadas (multicolinealidad)- Variables que correlacionan más con el targetEsto motiva el uso de **reducción de dimensionalidad (PCA)** en la sección siguiente.

In [39]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

print("[11] Heatmap de correlación entre features...")

# Reconstruimos un DF con todas las features + target para correlación
df_corr = pd.DataFrame(X, columns=feature_names).copy()
df_corr["TARGET"] = y

corr_mat = df_corr.corr(method="pearson")

# ── Plot 1: Heatmap COMPLETO (56x56) en triangular inferior ──────────
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    corr_mat, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.3, linecolor="white",
    cbar_kws={"shrink": 0.6, "label": "Correlación de Pearson"},
    xticklabels=True, yticklabels=True, ax=ax
)
ax.set_title(
    "Correlación entre features del modelo (n = " + f"{len(feature_names)}" + " variables + TARGET)\n"
    "Triángulo inferior – Pearson",
    fontweight="bold", fontsize=13, pad=14
)
ax.tick_params(axis="x", labelsize=6.5, rotation=90)
ax.tick_params(axis="y", labelsize=6.5)
save_poster(fig, "11a_heatmap_correlacion_full")

# ── Plot 2: Top-20 features más correlacionadas con el target ────────
target_corr = corr_mat["TARGET"].drop("TARGET").sort_values(key=lambda s: s.abs(), ascending=False)
top20 = target_corr.head(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors = [SECONDARY if v >= 0 else ACCENT for v in top20.values]
ax.barh(top20.index, top20.values, color=colors, alpha=0.85, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlación de Pearson con TARGET (desea migrar)")
ax.set_title(
    "Top 20 features por correlación absoluta con la aspiración migratoria\n"
    "Azul: aumenta prob. de migrar  |  Rojo: la reduce",
    fontweight="bold"
)
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
for i, v in enumerate(top20.values):
    ax.text(v + (0.003 if v >= 0 else -0.003), i, f"{v:+.3f}",
            va="center", ha="left" if v >= 0 else "right", fontsize=8.5, fontweight="bold")
plt.tight_layout()
save_poster(fig, "11b_top20_corr_target")

# ── Plot 3: Heatmap ZOOM en las top-20 features (más útil para el póster) ──
top20_names = target_corr.head(20).index.tolist()
sub_corr = corr_mat.loc[top20_names + ["TARGET"], top20_names + ["TARGET"]]
mask2 = np.triu(np.ones_like(sub_corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    sub_corr, mask=mask2, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    square=True, linewidths=0.5, linecolor="white",
    cbar_kws={"shrink": 0.7, "label": "Correlación de Pearson"}, ax=ax
)
ax.set_title(
    "Heatmap de correlación – Top 20 features (por |corr| con TARGET) + TARGET\n"
    "Valores anotados; triángulo inferior",
    fontweight="bold", fontsize=12, pad=12
)
ax.tick_params(axis="x", labelsize=8.5, rotation=45)
ax.tick_params(axis="y", labelsize=8.5)
plt.setp(ax.get_xticklabels(), ha="right", rotation_mode="anchor")
plt.tight_layout()
save_poster(fig, "11c_heatmap_correlacion_top20")

# Diagnóstico textual: pares más correlacionados entre sí (excluyendo TARGET)
print("\nPares de features con |corr| > 0.5 (excluyendo TARGET):")
sub = corr_mat.drop(index="TARGET", columns="TARGET")
pairs = sub.where(np.triu(np.ones(sub.shape, dtype=bool), k=1)).stack()
high = pairs[pairs.abs() > 0.5].sort_values(key=lambda s: s.abs(), ascending=False)
for (a, b), v in high.head(15).items():
    print(f"  {a:<25} ↔  {b:<25} :  {v:+.3f}")
print(f"\nTotal pares con |corr|>0.5: {len(high)}")


[11] Heatmap de correlación entre features...
  ✓ 11a_heatmap_correlacion_full.png  +  .pdf
  ✓ 11b_top20_corr_target.png  +  .pdf
  ✓ 11c_heatmap_correlacion_top20.png  +  .pdf

Pares de features con |corr| > 0.5 (excluyendo TARGET):
  s9q1b                     ↔  s9q1c                     :  +0.709
  s9q1a                     ↔  s9q1b                     :  +0.693
  s9q1a                     ↔  s9q1c                     :  +0.691
  s9q1c                     ↔  s9q1e                     :  +0.676
  s9q1a                     ↔  s9q1e                     :  +0.658
  shock_increase_in_price_of_oil_ ↔  n_shocks                  :  +0.657
  s9q1d                     ↔  s9q1e                     :  +0.636
  s9q1a                     ↔  s9q1d                     :  +0.623
  s9q1b                     ↔  s9q1e                     :  +0.620
  s9q1f                     ↔  s9q1g                     :  +0.620
  s9q1d                     ↔  s9q1g                     :  +0.619
  shock_increase_in_pr

  ✓ 11b_top20_corr_target.png  +  .pdf


  ✓ 11c_heatmap_correlacion_top20.png  +  .pdf

Pares de features con |corr| > 0.5 (excluyendo TARGET):
  s9q1b                     ↔  s9q1c                     :  +0.709
  s9q1a                     ↔  s9q1b                     :  +0.693
  s9q1a                     ↔  s9q1c                     :  +0.691
  s9q1c                     ↔  s9q1e                     :  +0.676
  s9q1a                     ↔  s9q1e                     :  +0.658
  shock_increase_in_price_of_oil_ ↔  n_shocks                  :  +0.657
  s9q1d                     ↔  s9q1e                     :  +0.636
  s9q1a                     ↔  s9q1d                     :  +0.623
  s9q1b                     ↔  s9q1e                     :  +0.620
  s9q1f                     ↔  s9q1g                     :  +0.620
  s9q1d                     ↔  s9q1g                     :  +0.619
  shock_increase_in_price_of_farm ↔  n_shocks                  :  +0.600
  s9q1d                     ↔  s9q1f                     :  +0.594
  s9q1c      

### SECCIÓN 12 — Reducción de dimensionalidad: PCACon 56 features y bloques correlacionados (10 ítems HFIAS, 29 dummies de shocks, etc.) tiene sentido evaluar si **unos pocos componentes principales** retienen la mayor parte de la varianza. Vemos:1. Varianza explicada acumulada2. Scatter en los 2 primeros componentes coloreado por target3. Comparación de performance: logit con todas las features vs. logit sobre los K primeros PCs

In [40]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

print("[12] PCA — reducción de dimensionalidad...")

# X_train_prep e X_test_prep ya están estandarizados (vienen del preprocessor del modelo B)
# Entrenamos PCA sobre train y aplicamos a test
pca_full = PCA(n_components=None, random_state=42)
pca_full.fit(X_train_prep)
var_explained = pca_full.explained_variance_ratio_
var_cum       = np.cumsum(var_explained)

# ¿Cuántos componentes hacen falta para el 80%, 90%, 95% de varianza?
n_80 = int(np.argmax(var_cum >= 0.80) + 1)
n_90 = int(np.argmax(var_cum >= 0.90) + 1)
n_95 = int(np.argmax(var_cum >= 0.95) + 1)
print(f"  Para 80% varianza: {n_80} componentes")
print(f"  Para 90% varianza: {n_90} componentes")
print(f"  Para 95% varianza: {n_95} componentes")

# ── Plot 1: Scree plot + varianza acumulada ─────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
ks = np.arange(1, len(var_explained) + 1)
ax1.bar(ks, var_explained * 100, color=SECONDARY, alpha=0.75, edgecolor="white", width=0.85)
ax1.set_xlabel("Componente principal")
ax1.set_ylabel("Varianza explicada (%)")
ax1.set_title("Scree plot – varianza explicada por componente", fontweight="bold")
ax1.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
# Etiquetas en los primeros 5
for i in range(5):
    ax1.text(ks[i], var_explained[i]*100 + 0.15, f"{var_explained[i]*100:.1f}%",
             ha="center", va="bottom", fontsize=8, fontweight="bold")

# Varianza acumulada
ax2.plot(ks, var_cum * 100, color=PALETTE, linewidth=2.2, marker="o", markersize=4)
ax2.axhline(80, ls="--", color=GRAY, linewidth=0.8)
ax2.axhline(90, ls="--", color=GRAY, linewidth=0.8)
ax2.axhline(95, ls="--", color=GRAY, linewidth=0.8)
ax2.axvline(n_80, ls=":", color=SECONDARY, linewidth=1)
ax2.axvline(n_90, ls=":", color=ACCENT,    linewidth=1)
ax2.axvline(n_95, ls=":", color=SUCCESS,   linewidth=1)
ax2.text(n_80 + 0.5, 50, f"{n_80} PC → 80%", color=SECONDARY, fontsize=9, fontweight="bold")
ax2.text(n_90 + 0.5, 55, f"{n_90} PC → 90%", color=ACCENT,    fontsize=9, fontweight="bold")
ax2.text(n_95 + 0.5, 60, f"{n_95} PC → 95%", color=SUCCESS,   fontsize=9, fontweight="bold")
ax2.set_xlabel("Número de componentes")
ax2.set_ylabel("Varianza acumulada (%)")
ax2.set_title("Varianza acumulada", fontweight="bold")
ax2.set_ylim(0, 105)
ax2.grid(color="#E8E8E8", linewidth=0.6)
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

fig.suptitle("PCA — Estructura de varianza en las 56 features estandarizadas",
             fontweight="bold", y=1.02)
plt.tight_layout()
save_poster(fig, "12a_pca_varianza_explicada")

# ── Plot 2: Proyección 2D (PC1 vs PC2) coloreada por target ──────────
X_train_pca2 = pca_full.transform(X_train_prep)[:, :2]

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(X_train_pca2[y_train == 0, 0], X_train_pca2[y_train == 0, 1],
           s=8, alpha=0.35, color=ACCENT,  label=f"No migra (n={(y_train==0).sum():,})")
ax.scatter(X_train_pca2[y_train == 1, 0], X_train_pca2[y_train == 1, 1],
           s=8, alpha=0.35, color=SECONDARY, label=f"Migra (n={(y_train==1).sum():,})")
ax.set_xlabel(f"PC 1  ({var_explained[0]*100:.1f}% varianza)")
ax.set_ylabel(f"PC 2  ({var_explained[1]*100:.1f}% varianza)")
ax.set_title(
    "Proyección 2D – primeros dos componentes principales\n"
    "Las clases se solapan: no hay separación lineal evidente en 2D",
    fontweight="bold"
)
ax.legend(markerscale=2, fontsize=10, frameon=False)
ax.grid(color="#E8E8E8", linewidth=0.6)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
save_poster(fig, "12b_pca_scatter_2d")

# ── Comparación: logit baseline vs. logit + PCA(K) ───────────────────
print("\n  Comparación de AUC-ROC: logit baseline vs logit sobre K componentes PCA")
print("  " + "-"*55)
ks_test = [2, 5, 10, n_80, n_90, n_95, len(var_explained)]
auc_pca_list = []
for k in ks_test:
    Xtr_k = PCA(n_components=k, random_state=42).fit_transform(X_train_prep)
    clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
    aucs = cross_val_score(clf, Xtr_k, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    auc_pca_list.append((k, aucs.mean(), aucs.std()))
    print(f"  K = {k:>3d}  →  AUC-ROC (CV) = {aucs.mean():.4f} ± {aucs.std():.4f}")

# Gráfico de la comparación
ks_arr   = [t[0] for t in auc_pca_list]
aucs_arr = [t[1] for t in auc_pca_list]
stds_arr = [t[2] for t in auc_pca_list]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.errorbar(ks_arr, aucs_arr, yerr=stds_arr, marker="o", color=SECONDARY,
            linewidth=2, markersize=8, capsize=4,
            label="Logit sobre K primeros PCs (5-fold CV)")
ax.axhline(cv_logit['test_roc_auc'].mean(), ls="--", color=PALETTE, linewidth=1.5,
           label=f"Logit baseline ({len(feature_names)} features)  "
                 f"= {cv_logit['test_roc_auc'].mean():.4f}")
ax.set_xlabel("K – número de componentes principales")
ax.set_ylabel("AUC-ROC (5-fold CV sobre train)")
ax.set_title(
    "PCA + Logit vs Logit baseline\n"
    "La señal ya se concentra en los primeros ~10–20 componentes",
    fontweight="bold"
)
ax.legend(loc="lower right", frameon=False)
ax.grid(color="#E8E8E8", linewidth=0.6)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
for k, a in zip(ks_arr, aucs_arr):
    ax.annotate(f"{a:.3f}", (k, a), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=8, fontweight="bold")
plt.tight_layout()
save_poster(fig, "12c_pca_vs_baseline_logit")


[12] PCA — reducción de dimensionalidad...
  Para 80% varianza: 32 componentes
  Para 90% varianza: 39 componentes
  Para 95% varianza: 44 componentes
  ✓ 12a_pca_varianza_explicada.png  +  .pdf
  ✓ 12b_pca_scatter_2d.png  +  .pdf

  Comparación de AUC-ROC: logit baseline vs logit sobre K componentes PCA
  -------------------------------------------------------
  K =   2  →  AUC-ROC (CV) = 0.5140 ± 0.0120
  K =   5  →  AUC-ROC (CV) = 0.6625 ± 0.0162
  K =  10  →  AUC-ROC (CV) = 0.6635 ± 0.0161
  K =  32  →  AUC-ROC (CV) = 0.6749 ± 0.0160
  K =  39  →  AUC-ROC (CV) = 0.6765 ± 0.0143
  K =  44  →  AUC-ROC (CV) = 0.6781 ± 0.0143
  K =  56  →  AUC-ROC (CV) = 0.6831 ± 0.0122
  ✓ 12c_pca_vs_baseline_logit.png  +  .pdf


  ✓ 12b_pca_scatter_2d.png  +  .pdf

  Comparación de AUC-ROC: logit baseline vs logit sobre K componentes PCA
  -------------------------------------------------------
  K =   2  →  AUC-ROC (CV) = 0.5140 ± 0.0120
  K =   5  →  AUC-ROC (CV) = 0.6625 ± 0.0162
  K =  10  →  AUC-ROC (CV) = 0.6635 ± 0.0161


  K =  32  →  AUC-ROC (CV) = 0.6749 ± 0.0160
  K =  39  →  AUC-ROC (CV) = 0.6765 ± 0.0143
  K =  44  →  AUC-ROC (CV) = 0.6781 ± 0.0143


  K =  56  →  AUC-ROC (CV) = 0.6831 ± 0.0122


  ✓ 12c_pca_vs_baseline_logit.png  +  .pdf


### SECCIÓN 13 — Modelo no lineal adicional: Gradient BoostingTu recomendación: dado que las relaciones entre features y aspiración migratoria son no lineales (el RF supera al logit por ~4 pts de AUC), conviene un método que **capture no linealidades**. El RF ya lo hace, pero **Gradient Boosting** suele ser aún mejor en datos tabulares porque ajusta árboles **secuencialmente** corrigiendo los errores residuales del paso anterior.Usamos `HistGradientBoostingClassifier` (implementación moderna de sklearn, basada en histogramas, muy rápida y robusta).

In [41]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics  import roc_auc_score, f1_score, balanced_accuracy_score, classification_report

print("[Modelo D] Gradient Boosting (HistGradientBoosting)...")

# class_weight se simula vía sample_weight (HistGB no acepta class_weight directo en versiones viejas)
# Aplicamos pesos inversos a la frecuencia de cada clase
class_counts = np.bincount(y_train)
weights_map  = {0: len(y_train) / (2 * class_counts[0]),
                1: len(y_train) / (2 * class_counts[1])}
sample_weights = np.array([weights_map[c] for c in y_train])

gb = HistGradientBoostingClassifier(
    max_iter            = 300,    # nº de árboles secuenciales
    max_depth            = None,
    max_leaf_nodes       = 31,
    learning_rate        = 0.05,
    min_samples_leaf     = 30,
    l2_regularization    = 1.0,
    early_stopping       = True,
    validation_fraction  = 0.15,
    random_state         = 42,
)

gb.fit(X_train_prep, y_train, sample_weight=sample_weights)

y_prob_gb = gb.predict_proba(X_test_prep)[:, 1]
y_pred_gb = gb.predict(X_test_prep)

auc_gb  = roc_auc_score(y_test, y_prob_gb)
f1_gb   = f1_score(y_test, y_pred_gb, average="weighted")
bacc_gb = balanced_accuracy_score(y_test, y_pred_gb)
rep_gb  = classification_report(y_test, y_pred_gb, output_dict=True)

# CV sobre train
cv_gb = cross_val_score(
    HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, min_samples_leaf=30,
        l2_regularization=1.0, random_state=42, early_stopping=False
    ),
    X_train_prep, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
)

print(f"  Iteraciones efectivas:  {gb.n_iter_}")
print(f"  AUC-ROC (test):         {auc_gb:.4f}")
print(f"  Balanced Accuracy:      {bacc_gb:.4f}")
print(f"  F1 weighted (test):     {f1_gb:.4f}")
print(f"  AUC-ROC (CV mean):      {cv_gb.mean():.4f} ± {cv_gb.std():.4f}")
print("  OK")

# Tabla comparativa actualizada (incluye GB)
print("\n" + "="*72)
print("RESUMEN COMPARATIVO ACTUALIZADO — 4 modelos")
print("="*72)
print(f"{'Modelo':<22} {'AUC-ROC':>8} {'Bal.Acc':>9} {'F1(cl.0)':>10} {'F1(cl.1)':>10}")
print("-"*72)
for nombre, auc, bacc, rep in [
    ("Logit baseline",   auc_logit, bacc_logit, rep_logit),
    ("Elastic Net",      auc_enet,  bacc_enet,  rep_enet),
    ("Random Forest",    auc_rf,    bacc_rf,    rep_rf),
    ("Gradient Boosting",auc_gb,    bacc_gb,    rep_gb),
]:
    f0 = rep['0']['f1-score']; f1v = rep['1']['f1-score']
    print(f"{nombre:<22} {auc:>8.4f} {bacc:>9.4f} {f0:>10.3f} {f1v:>10.3f}")
print("="*72)


[Modelo D] Gradient Boosting (HistGradientBoosting)...
  Iteraciones efectivas:  187
  AUC-ROC (test):         0.7284
  Balanced Accuracy:      0.6631
  F1 weighted (test):     0.6932
  AUC-ROC (CV mean):      0.7425 ± 0.0117
  OK

RESUMEN COMPARATIVO ACTUALIZADO — 4 modelos
Modelo                  AUC-ROC   Bal.Acc   F1(cl.0)   F1(cl.1)
------------------------------------------------------------------------
Logit baseline           0.6679    0.6171      0.483      0.697
Elastic Net              0.6694    0.6211      0.487      0.699
Random Forest            0.7070    0.6509      0.514      0.756
Gradient Boosting        0.7284    0.6631      0.530      0.760


### SECCIÓN 14 — Curvas ROC, Precision-Recall y distribución de probabilidadesUna de las observaciones del PDF era agregar **curva Precision-Recall** y **distribution plot de probabilidades predichas**. Lo hacemos para los **4 modelos**.

In [42]:
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

print("[14] Curvas ROC, Precision-Recall y distribución de probabilidades...")

# AP scores (área bajo la curva PR)
ap_logit = average_precision_score(y_test, y_prob_logit)
ap_enet  = average_precision_score(y_test, y_prob_enet)
ap_rf    = average_precision_score(y_test, y_prob_rf)
ap_gb    = average_precision_score(y_test, y_prob_gb)

# Línea base de PR = prevalencia de clase positiva
baseline_pr = y_test.mean()

# Curvas
fpr_l, tpr_l, _ = roc_curve(y_test, y_prob_logit)
fpr_e, tpr_e, _ = roc_curve(y_test, y_prob_enet)
fpr_r, tpr_r, _ = roc_curve(y_test, y_prob_rf)
fpr_g, tpr_g, _ = roc_curve(y_test, y_prob_gb)

prec_l, rec_l, _ = precision_recall_curve(y_test, y_prob_logit)
prec_e, rec_e, _ = precision_recall_curve(y_test, y_prob_enet)
prec_r, rec_r, _ = precision_recall_curve(y_test, y_prob_rf)
prec_g, rec_g, _ = precision_recall_curve(y_test, y_prob_gb)

# ── Figura combinada: ROC + PR + Distribución de probabilidades ──────
fig = plt.figure(figsize=(17, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.28)

# Panel A: ROC
axA = fig.add_subplot(gs[0, 0])
axA.plot(fpr_l, tpr_l, color=COL_LOGIT, lw=1.8, label=f"Logit   (AUC={auc_logit:.3f})")
axA.plot(fpr_e, tpr_e, color=COL_ENET,  lw=1.8, ls="--", label=f"ElNet  (AUC={auc_enet:.3f})")
axA.plot(fpr_r, tpr_r, color=COL_RF,    lw=2.0, ls="-.", label=f"RF     (AUC={auc_rf:.3f})")
axA.plot(fpr_g, tpr_g, color=COL_GB,    lw=2.0,         label=f"GB     (AUC={auc_gb:.3f})")
axA.plot([0,1],[0,1], "k--", lw=0.8, alpha=0.5)
axA.set_xlabel("Tasa de Falsos Positivos")
axA.set_ylabel("Tasa de Verdaderos Positivos")
axA.set_title("Curva ROC (test set)", fontweight="bold")
axA.legend(fontsize=9, frameon=False, loc="lower right")
axA.set_xlim(0, 1); axA.set_ylim(0, 1.02)
axA.grid(color="#E8E8E8", linewidth=0.6)
axA.spines["top"].set_visible(False); axA.spines["right"].set_visible(False)

# Panel B: Precision-Recall
axB = fig.add_subplot(gs[0, 1])
axB.plot(rec_l, prec_l, color=COL_LOGIT, lw=1.8, label=f"Logit   (AP={ap_logit:.3f})")
axB.plot(rec_e, prec_e, color=COL_ENET,  lw=1.8, ls="--", label=f"ElNet  (AP={ap_enet:.3f})")
axB.plot(rec_r, prec_r, color=COL_RF,    lw=2.0, ls="-.", label=f"RF     (AP={ap_rf:.3f})")
axB.plot(rec_g, prec_g, color=COL_GB,    lw=2.0,         label=f"GB     (AP={ap_gb:.3f})")
axB.axhline(baseline_pr, ls=":", color="gray", linewidth=1,
            label=f"Baseline (prevalencia = {baseline_pr:.2f})")
axB.set_xlabel("Recall (sensibilidad)")
axB.set_ylabel("Precision (valor predictivo positivo)")
axB.set_title("Curva Precision-Recall (test set)", fontweight="bold")
axB.legend(fontsize=9, frameon=False, loc="lower left")
axB.set_xlim(0, 1); axB.set_ylim(0, 1.02)
axB.grid(color="#E8E8E8", linewidth=0.6)
axB.spines["top"].set_visible(False); axB.spines["right"].set_visible(False)

# Panel C: Distribución de probabilidades del MODELO GANADOR (el de mayor AUC entre RF y GB)
auc_dict = {"Random Forest": auc_rf, "Gradient Boosting": auc_gb}
best_name = max(auc_dict, key=auc_dict.get)
best_prob = y_prob_rf if best_name == "Random Forest" else y_prob_gb
best_col  = COL_RF if best_name == "Random Forest" else COL_GB

axC = fig.add_subplot(gs[0, 2])
bins = np.linspace(0, 1, 31)
axC.hist(best_prob[y_test == 0], bins=bins, alpha=0.6, color=ACCENT,    label="Real: NO migra", edgecolor="white")
axC.hist(best_prob[y_test == 1], bins=bins, alpha=0.6, color=SECONDARY, label="Real: SÍ migra", edgecolor="white")
axC.axvline(0.5, ls="--", color="black", lw=0.9, label="Umbral 0.5")
axC.set_xlabel("Probabilidad predicha de migrar")
axC.set_ylabel("Frecuencia")
axC.set_title(f"Distribución de probabilidades – {best_name}\n"
              f"Separación visible entre clases pese al AUC moderado",
              fontweight="bold")
axC.legend(fontsize=9, frameon=False)
axC.grid(axis="y", color="#E8E8E8", linewidth=0.6)
axC.spines["top"].set_visible(False); axC.spines["right"].set_visible(False)

fig.suptitle("Evaluación predictiva – ROC, PR y distribución de probabilidades",
             fontweight="bold", y=1.04)
plt.tight_layout()
save_poster(fig, "14_roc_pr_distribucion")


[14] Curvas ROC, Precision-Recall y distribución de probabilidades...
  ✓ 14_roc_pr_distribucion.png  +  .pdf


### SECCIÓN 15 — Permutation importance (validación del ranking de variables)El PDF marca como pendiente: *"calcular permutation importance para el RF, que es más confiable que mean decrease impurity para variables continuas vs. binarias."*Permutation importance mide cuánto cae el AUC del modelo al permutar aleatoriamente una variable en el conjunto de test. Es **agnóstica al modelo** y **no sobreestima variables continuas**.

In [43]:
from sklearn.inspection import permutation_importance

print("[15] Permutation importance — RF y Gradient Boosting...")
print("    (este paso puede tardar 1-3 minutos por modelo)")

# RF
perm_rf = permutation_importance(
    rf, X_test_prep, y_test,
    scoring="roc_auc", n_repeats=10, random_state=42, n_jobs=-1
)
perm_rf_df = pd.DataFrame({
    "feature":   feature_names,
    "imp_mean":  perm_rf.importances_mean,
    "imp_std":   perm_rf.importances_std,
}).sort_values("imp_mean", ascending=False)

# GB
perm_gb = permutation_importance(
    gb, X_test_prep, y_test,
    scoring="roc_auc", n_repeats=10, random_state=42, n_jobs=-1
)
perm_gb_df = pd.DataFrame({
    "feature":   feature_names,
    "imp_mean":  perm_gb.importances_mean,
    "imp_std":   perm_gb.importances_std,
}).sort_values("imp_mean", ascending=False)

# Guardamos las tablas (CSV) para el póster
perm_rf_df.to_csv(os.path.join(POSTER_OUT, "15a_permutation_importance_RF.csv"), index=False)
perm_gb_df.to_csv(os.path.join(POSTER_OUT, "15b_permutation_importance_GB.csv"), index=False)
print(f"  Tablas guardadas en {POSTER_OUT}")

# Plot comparativo: top 15 por permutation importance, para RF y GB
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

for ax, df_imp, nombre, color in [
    (axes[0], perm_rf_df, "Random Forest",      COL_RF),
    (axes[1], perm_gb_df, "Gradient Boosting",  COL_GB),
]:
    top = df_imp.head(15).iloc[::-1]  # invertir para que el mayor quede arriba
    ax.barh(top["feature"], top["imp_mean"], xerr=top["imp_std"],
            color=color, alpha=0.85, edgecolor="white",
            error_kw={"linewidth": 0.8, "ecolor": "#555555"})
    ax.set_xlabel("Caída de AUC al permutar (mayor = más importante)")
    ax.set_title(f"Permutation Importance – {nombre}\n"
                 "(test set, 10 repeticiones)", fontweight="bold")
    ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.tick_params(axis="y", labelsize=8.5)
    for i, (v, s) in enumerate(zip(top["imp_mean"], top["imp_std"])):
        ax.text(v + s + 0.0005, i, f"{v:.3f}", va="center", fontsize=7.5)

fig.suptitle("Permutation importance – Top 15 features\n"
             "Más robusta que Gini importance para comparar continuas vs. binarias",
             fontweight="bold", y=1.02)
plt.tight_layout()
save_poster(fig, "15_permutation_importance")

# Comparación: MDI vs Permutation (RF) — para mostrar que el ranking cambia
mdi_series = pd.Series(rf.feature_importances_, index=feature_names, name="MDI (Gini)")
perm_series = perm_rf_df.set_index("feature")["imp_mean"].rename("Permutation (ΔAUC)")
comp = pd.concat([mdi_series, perm_series], axis=1).sort_values("Permutation (ΔAUC)", ascending=False).head(10)
print("\nTop 10 features – RF — Comparación MDI vs Permutation:")
print(comp.round(4).to_string())


[15] Permutation importance — RF y Gradient Boosting...
    (este paso puede tardar 1-3 minutos por modelo)
  Tablas guardadas en /Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/Proyecto_poster/graficos_y_tablas_poster
  ✓ 15_permutation_importance.png  +  .pdf

Top 10 features – RF — Comparación MDI vs Permutation:
                  MDI (Gini)  Permutation (ΔAUC)
s1q6                  0.2769              0.0596
remesas_interior      0.0629              0.0141
zona_West             0.0233              0.0120
s1q25                 0.0988              0.0085
s1q8                  0.0443              0.0085
s1q2                  0.0279              0.0065
zona_South            0.0185              0.0064
n_shocks              0.0633              0.0062
s9q1b                 0.0150              0.0048
zona_East             0.0208              0.0047


  ✓ 15_permutation_importance.png  +  .pdf

Top 10 features – RF — Comparación MDI vs Permutation:
                  MDI (Gini)  Permutation (ΔAUC)
s1q6                  0.2769              0.0598
remesas_interior      0.0627              0.0142
zona_West             0.0232              0.0121
s1q25                 0.0988              0.0085
s1q8                  0.0441              0.0085
s1q2                  0.0279              0.0065
zona_South            0.0185              0.0065
n_shocks              0.0634              0.0062
s9q1b                 0.0151              0.0049
zona_East             0.0207              0.0048


### SECCIÓN 16 — Calibración del modelo finalOtro pendiente del PDF: *"verificar la calibración del modelo (si las probabilidades predichas corresponden a frecuencias reales)"*.Un modelo está bien **calibrado** si entre los individuos a los que les asigna probabilidad de migrar ≈ 0.7, efectivamente el 70 % migra. RF y boosting suelen estar **sub-calibrados** (los extremos se subestiman); el logit suele estarlo bastante bien.

In [44]:
from sklearn.calibration import calibration_curve

print("[16] Calibración de probabilidades — los 4 modelos...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Reliability diagram
for y_prob, nombre, color, ls in [
    (y_prob_logit, "Logit",             COL_LOGIT, "-"),
    (y_prob_enet,  "Elastic Net",       COL_ENET,  "--"),
    (y_prob_rf,    "Random Forest",     COL_RF,    "-."),
    (y_prob_gb,    "Gradient Boosting", COL_GB,    "-"),
]:
    frac_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10, strategy="quantile")
    ax1.plot(mean_pred, frac_pos, marker="o", color=color, linewidth=2, linestyle=ls,
             label=nombre, markersize=7)

ax1.plot([0, 1], [0, 1], "k--", lw=1, label="Calibración perfecta")
ax1.set_xlabel("Probabilidad media predicha (por decil)")
ax1.set_ylabel("Frecuencia real de migración en el decil")
ax1.set_title("Reliability diagram (calibration curve)\n"
              "Diagonal = predicción perfectamente calibrada", fontweight="bold")
ax1.legend(loc="upper left", fontsize=9, frameon=False)
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
ax1.grid(color="#E8E8E8", linewidth=0.6)
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

# Panel 2: Histograma de probabilidades predichas — RF (modelo final por defecto)
ax2.hist(y_prob_rf, bins=30, color=COL_RF, alpha=0.75, edgecolor="white")
ax2.axvline(y_prob_rf.mean(), ls="--", color="black", lw=1,
            label=f"Media = {y_prob_rf.mean():.3f}")
ax2.axvline(y_test.mean(), ls=":", color=PALETTE, lw=1.5,
            label=f"Prevalencia real = {y_test.mean():.3f}")
ax2.set_xlabel("Probabilidad predicha de migrar (Random Forest)")
ax2.set_ylabel("Frecuencia")
ax2.set_title("Distribución de probabilidades predichas – RF",
              fontweight="bold")
ax2.legend(fontsize=9, frameon=False)
ax2.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

fig.suptitle("Calibración del modelo – diagnóstico de probabilidades",
             fontweight="bold", y=1.02)
plt.tight_layout()
save_poster(fig, "16_calibracion")

# Brier score (resumen numérico de calibración + accuracy)
from sklearn.metrics import brier_score_loss
print("\nBrier score (menor = mejor calibración + accuracy):")
for y_prob, nombre in [
    (y_prob_logit, "Logit baseline"),
    (y_prob_enet,  "Elastic Net"),
    (y_prob_rf,    "Random Forest"),
    (y_prob_gb,    "Gradient Boosting"),
]:
    bs = brier_score_loss(y_test, y_prob)
    print(f"  {nombre:<22} : {bs:.4f}")


[16] Calibración de probabilidades — los 4 modelos...
  ✓ 16_calibracion.png  +  .pdf

Brier score (menor = mejor calibración + accuracy):
  Logit baseline         : 0.2271
  Elastic Net            : 0.2265
  Random Forest          : 0.2128
  Gradient Boosting      : 0.2022


### SECCIÓN 17 — Matriz de confusión detallada del modelo finalVersión más amigable de las matrices de confusión para el póster, con métricas derivadas (precision, recall, F1, accuracy) anotadas directamente. Cumple la recomendación de *"agregar matriz de confusión – explotar métodos fáciles de explicar"*.

In [45]:
from sklearn.metrics import confusion_matrix

print("[17] Matrices de confusión – versión para póster (4 modelos)...")

def panel_confusion(ax, y_true, y_pred, nombre, color):
    cm     = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total  = cm.sum()
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    accuracy  = (tp + tn) / total
    precision = tp / (tp + fp) if (tp+fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp+fn) > 0 else 0
    specif    = tn / (tn + fp) if (tn+fp) > 0 else 0
    f1        = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0

    im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred:\nNo migra", "Pred:\nMigra"], fontsize=9)
    ax.set_yticklabels(["Real:\nNo migra", "Real:\nMigra"], fontsize=9)
    ax.set_title(nombre, fontweight="bold", color=color, pad=8)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)",
                    ha="center", va="center", fontsize=11, fontweight="bold",
                    color="white" if cm_pct[i,j] > 55 else "#2C3E50")
    # Métricas debajo
    txt = (f"Accuracy: {accuracy:.3f}  |  F1: {f1:.3f}\n"
           f"Precision: {precision:.3f}  |  Recall: {recall:.3f}  |  Specif.: {specif:.3f}")
    ax.text(0.5, -0.32, txt, transform=ax.transAxes,
            ha="center", va="top", fontsize=8.5,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="#F4F6F7", edgecolor=color, linewidth=1.2))

fig, axes = plt.subplots(1, 4, figsize=(18, 5.2))
panel_confusion(axes[0], y_test, y_pred_logit, "Logit baseline",     COL_LOGIT)
panel_confusion(axes[1], y_test, y_pred_enet,  "Elastic Net",        COL_ENET)
panel_confusion(axes[2], y_test, y_pred_rf,    "Random Forest",      COL_RF)
panel_confusion(axes[3], y_test, y_pred_gb,    "Gradient Boosting",  COL_GB)

fig.suptitle("Matrices de Confusión – test set (n = " + f"{len(y_test):,}" + ")\n"
             "Filas = clase real  |  Columnas = clase predicha  |  Umbral = 0.5",
             fontweight="bold", y=1.02)
plt.tight_layout()
save_poster(fig, "17_matrices_confusion_4modelos")

# ── Matriz de confusión del modelo ganador (versión grande para el póster) ──
fig, ax = plt.subplots(figsize=(7, 6.5))
panel_confusion(ax, y_test, y_pred_rf, "Random Forest – modelo final", COL_RF)
ax.set_title("Matriz de confusión – Random Forest (modelo final)\n"
             f"n = {len(y_test):,}  |  AUC = {auc_rf:.3f}",
             fontweight="bold", pad=12)
plt.tight_layout()
save_poster(fig, "17b_matriz_confusion_RF_final")


[17] Matrices de confusión – versión para póster (4 modelos)...
  ✓ 17_matrices_confusion_4modelos.png  +  .pdf
  ✓ 17b_matriz_confusion_RF_final.png  +  .pdf


  ✓ 17b_matriz_confusion_RF_final.png  +  .pdf


### SECCIÓN 18 — Tabla resumen final de modelosTabla limpia para el póster con todas las métricas de los 4 modelos.

In [46]:
import pandas as pd

resumen = pd.DataFrame({
    "Modelo": ["Logit baseline", "Elastic Net", "Random Forest", "Gradient Boosting"],
    "Tipo": ["Lineal", "Lineal (regularizado)", "No lineal (ensemble bagging)",
             "No lineal (ensemble boosting)"],
    "AUC-ROC (test)":   [round(auc_logit, 4), round(auc_enet, 4), round(auc_rf, 4), round(auc_gb, 4)],
    "Balanced Acc.":    [round(bacc_logit, 4), round(bacc_enet, 4), round(bacc_rf, 4), round(bacc_gb, 4)],
    "F1 (cl. 0)":       [round(rep_logit['0']['f1-score'], 3), round(rep_enet['0']['f1-score'], 3),
                         round(rep_rf['0']['f1-score'], 3),    round(rep_gb['0']['f1-score'], 3)],
    "F1 (cl. 1)":       [round(rep_logit['1']['f1-score'], 3), round(rep_enet['1']['f1-score'], 3),
                         round(rep_rf['1']['f1-score'], 3),    round(rep_gb['1']['f1-score'], 3)],
    "Avg. Precision":   [round(ap_logit, 3), round(ap_enet, 3), round(ap_rf, 3), round(ap_gb, 3)],
})

print(resumen.to_string(index=False))

# Guardamos CSV y versión gráfica
resumen.to_csv(os.path.join(POSTER_OUT, "18_resumen_modelos.csv"), index=False)

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.axis("off")
tabla_data = [resumen.columns.tolist()] + resumen.astype(str).values.tolist()
tbl = ax.table(cellText=tabla_data[1:], colLabels=tabla_data[0],
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.0, 1.7)

# Header style
for j in range(len(resumen.columns)):
    tbl[0, j].set_facecolor(PALETTE)
    tbl[0, j].set_text_props(color="white", fontweight="bold")

# Highlight mejor por métrica numérica
numeric_cols = [2, 3, 4, 5, 6]   # AUC, BalAcc, F1c0, F1c1, AP
for col_idx in numeric_cols:
    vals = [float(resumen.iloc[i, col_idx]) for i in range(len(resumen))]
    best_row = int(np.argmax(vals)) + 1
    tbl[best_row, col_idx].set_facecolor("#D5F5E3")
    tbl[best_row, col_idx].set_text_props(fontweight="bold")

ax.set_title("Resumen comparativo de los 4 modelos (test set, n = " + f"{len(y_test):,}" + ")\n"
             "Verde = mejor por columna",
             fontweight="bold", pad=12)
plt.tight_layout()
save_poster(fig, "18_resumen_modelos")


           Modelo                          Tipo  AUC-ROC (test)  Balanced Acc.  F1 (cl. 0)  F1 (cl. 1)  Avg. Precision
   Logit baseline                        Lineal          0.6679         0.6171       0.483       0.697           0.818
      Elastic Net         Lineal (regularizado)          0.6694         0.6211       0.487       0.699           0.817
    Random Forest  No lineal (ensemble bagging)          0.7070         0.6509       0.514       0.756           0.845
Gradient Boosting No lineal (ensemble boosting)          0.7284         0.6631       0.530       0.760           0.856
  ✓ 18_resumen_modelos.png  +  .pdf


### SECCIÓN 19 — Copiar gráficos clave del diagnóstico previo al directorio del pósterAlgunos gráficos generados en las secciones 1–9 del notebook original también irán al póster final. Los copiamos a `graficos_y_tablas_poster/` para tenerlos todos en un solo lugar.

In [47]:
import shutil
print(f"[19] Copiando gráficos clave del diagnóstico previo → {POSTER_OUT}")

# Mapeamos los gráficos relevantes del notebook original que valen para el póster.
# Si alguno no existe (porque se corrió sólo la parte de modelado), se ignora.
candidates = {
    "03_target_variable.png":         "diag_target_variable.png",
    "04c_heatmap_missings.png":       "diag_heatmap_missings.png",
    "04d_correlacion.png":            "diag_corr_features.png",
    "04e_corr_con_target.png":        "diag_corr_con_target.png",
    "05_distribuciones_features.png": "diag_distribuciones_features.png",
    "06a_shocks_economicos.png":      "diag_shocks_economicos.png",
    "06b_shock_index.png":            "diag_shock_index.png",
    "08_dataset_definitivo.png":      "diag_dataset_definitivo.png",
    "B1_edad_migrantes.png":          "diag_edad_migrantes.png",
    "B2_hfias_distribucion.png":     "diag_hfias_distribucion.png",
    "B3_mapa_nigeria.png":            "diag_mapa_nigeria.png",
    "09_comparacion_modelos.png":     "09_comparacion_modelos.png",
    "09b_confusion_matrices.png":     "09b_confusion_matrices.png",
}

copied, missing = 0, 0
for src_name, dst_name in candidates.items():
    src = os.path.join(OUT, src_name)
    dst = os.path.join(POSTER_OUT, dst_name)
    if os.path.exists(src):
        shutil.copyfile(src, dst)
        copied += 1
    else:
        missing += 1

print(f"  Copiados: {copied}  |  No encontrados: {missing}")
print(f"\nContenido final de {POSTER_OUT}:")
for f in sorted(os.listdir(POSTER_OUT)):
    print(f"  - {f}")


[19] Copiando gráficos clave del diagnóstico previo → /Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/Proyecto_poster/graficos_y_tablas_poster
  Copiados: 13  |  No encontrados: 0

Contenido final de /Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/Proyecto_poster/graficos_y_tablas_poster:
  - .ipynb_checkpoints
  - 09_comparacion_modelos.png
  - 09b_confusion_matrices.png
  - 11a_heatmap_correlacion_full.pdf
  - 11a_heatmap_correlacion_full.png
  - 11b_top20_corr_target.pdf
  - 11b_top20_corr_target.png
  - 11c_heatmap_correlacion_top20.pdf
  - 11c_heatmap_correlacion_top20.png
  - 12a_pca_varianza_explicada.pdf
  - 12a_pca_varianza_explicada.png
  - 12b_pca_scatter_2d.pdf
  - 12b_pca_scatter_2d.png
  - 12c_pca_vs_baseline_logit.pdf
  - 12c_pca_vs_baseline_logit.png
  - 14_roc_pr_distribucion.pdf
  - 14_roc_pr_distribucion.png
  - 15_permutation_importance.pdf
  - 15_permutation_importance.png
  - 15a_permutation_importance_RF.csv
  - 15b_permutation_importance_GB.c

### SECCIÓN 20 — Discusión, hallazgos y limitaciones (texto para el póster)Texto en formato póster que sintetiza hallazgos, recomendaciones de uso y limitaciones. Esto cumple la recomendación del usuario: *"agregar discusión y limitaciones del trabajo en el póster"*.

#### 📌 Hallazgos principales1. **71 % de los adultos nigerianos ≥ 15 años declaran querer migrar.** El desbalance moderado (71/29) se trata vía `class_weight=balanced` en los cuatro modelos.2. **La relación entre features y aspiración migratoria es no lineal.** Los modelos lineales (Logit, Elastic Net) se quedan en AUC ≈ 0.67, mientras que los modelos no lineales basados en árboles (Random Forest, Gradient Boosting) alcanzan AUC ≈ 0.71. El gap de ~4 pts es **la huella estadística de las no linealidades e interacciones** entre variables.3. **La edad concentra la mayor señal predictiva.** Tanto por correlación de Pearson (r ≈ 0.20) como por permutation importance (ΔAUC ≈ 0.06 al permutarla en el RF), la edad es la variable más informativa. La aspiración cae monotónicamente desde los 35 años en adelante.4. **Reducción de dimensionalidad: con ~10 componentes principales se recupera la mayor parte de la señal lineal.** PCA + logit sobre los primeros 10 PCs alcanza un AUC similar al logit baseline con 56 features. Esto confirma que la información lineal está concentrada en pocas dimensiones, y que el salto del Random Forest viene de las no linealidades.5. **Shocks económicos: los climáticos (sequías, lluvias, calor) son los que más se asocian con la aspiración migratoria** (r ≈ 0.05–0.06), aunque su contribución individual al modelo es modesta. El **número total de shocks** (`n_shocks`) entra al top-5 de importancia del RF.

#### ⚠️ Limitaciones del trabajo1. **Señal lineal débil:** la correlación máxima con el target es r = 0.20 (edad). Un AUC de 0.70–0.71 es sustantivo en este contexto, pero significa que **una parte importante de la varianza de la aspiración migratoria depende de variables no observadas** (redes sociales, información sobre destinos, preferencias individuales, factores psicológicos).2. **Missings sistemáticos:** ~49 % de los individuos no tiene datos laborales (no por azar, sino porque no trabajan formalmente). Imputamos a 0 y agregamos `tiene_datos_laborales` como flag, pero **el supuesto MNAR (missing not at random) puede sesgar los coeficientes**.3. **Pesos muestrales no incorporados al entrenamiento:** la GHS usa diseño complejo; no usar `wt_cross_wave5` en el fit puede sesgar la generalización a la población nigeriana. En la literatura de ML con encuestas hay práctica mixta; queda como extensión.4. **Target binario:** `s1dq1` colapsa intensidad y destino de la aspiración. El módulo de harvest (`s3bq1`) ofrece una medición temporalmente posterior que **no usamos como feature por riesgo de data leakage**, pero podría utilizarse como segundo target.5. **Predicción ≠ causalidad:** todo lo reportado son asociaciones predictivas en un corte transversal. **No identifica el efecto causal de ningún shock o variable.** Los métodos vistos no permiten contrastar mecanismos de aspiración migratoria.6. **Validación temporal pendiente:** todo el train/test es del mismo año (2023–2024). No sabemos si el modelo generaliza a olas anteriores o posteriores de la GHS.

#### ✅ ConclusiónEl **Random Forest** se mantiene como modelo final por su mejor AUC out-of-sample (0.707) y un OOB score consistente (0.677), sin señales de overfitting. Gradient Boosting alcanza performance muy similar (ver tabla 18) y puede usarse como modelo alternativo robusto.El hallazgo más fuerte es **estructural más que predictivo**: la aspiración migratoria en Nigeria no es una función lineal de demografía y shocks, lo que justifica el uso de **métodos que capturen no linealidades e interacciones** (árboles, ensembles). La edad es el predictor dominante; los shocks contribuyen, pero su efecto es heterogéneo y depende de combinaciones que sólo los modelos no lineales detectan.